# The Look Ecommerce — Build and Validate Pipeline

## Goal

Build the analytical database progressively and validate every layer.

Current modules:

1. `01_staging.sql`
2. `02_core_dimensions.sql`

The notebook will later include facts, marts, QA, exports, and advanced analysis.

In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd
from IPython.display import display

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

print("Python:", sys.version)
print("DuckDB:", duckdb.__version__)
print("pandas:", pd.__version__)

Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
DuckDB: 1.5.5
pandas: 3.0.5


## 1. Project configuration

Define the project, source-data, database, source-code, and SQL locations.

In [3]:
project_root = Path(
    r"C:\Users\phanh\data_analysis\p_projects"
    r"\the_look_ecommerce\practice_analysis"
)

source_csv_dir = Path(
    r"C:\Users\phanh\data_analysis\p_projects"
    r"\the_look_ecommerce\original_analysis"
    r"\5_the_look_ecommerce\csv_version"
    r"\3_thelookecommerce"
)

database_path = (
    project_root
    / "artifacts"
    / "practice_analytics.duckdb"
)

sql_dir = (
    project_root
    / "sql"
    / "duckdb"
)

src_dir = project_root / "src"

print("Project:", project_root)
print("Source:", source_csv_dir)
print("Database:", database_path)
print("SQL folder:", sql_dir)

Project: C:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis
Source: C:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\original_analysis\5_the_look_ecommerce\csv_version\3_thelookecommerce
Database: C:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\artifacts\practice_analytics.duckdb
SQL folder: C:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\sql\duckdb


In [4]:
#validate required files
source_tables = [
    "users",
    "products",
    "orders",
    "order_items",
    "events",
    "inventory_events",
    "distribution_centers",
]

required_csv_files = [
    source_csv_dir / f"{table}.csv"
    for table in source_tables
]

required_sql_files = [
    sql_dir / "01_staging.sql",
    sql_dir / "02_core_dimensions.sql",
    sql_dir / "03_core_facts.sql",
    sql_dir / "04_mart_acquisition_funnel.sql",
    sql_dir / "05_mart_commercial_customer.sql",
    sql_dir / "06_mart_operations_inventory.sql",
]

file_checks = []

for path in required_csv_files + required_sql_files:
    file_checks.append(
        {
            "path": str(path),
            "exists": path.exists(),
        }
    )

file_check_results = pd.DataFrame(file_checks)

display(file_check_results)

,path,exists
0,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
1,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
2,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
3,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
4,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
5,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
6,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
7,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
8,C:\Users\phanh\data_analysis\p_projects\the_lo...,True
9,C:\Users\phanh\data_analysis\p_projects\the_lo...,True


## 2. Load the pipeline utilities

Import `ingest_raw()` from `src/pipeline.py`.

In [5]:
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from pipeline import ingest_raw

print("ingest_raw imported successfully.")

ingest_raw imported successfully.


In [6]:
#open DuckDB
if "connection" in globals():
    try:
        connection.close()
    except Exception:
        pass

database_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

connection = duckdb.connect(
    str(database_path)
)

connection.execute("SET threads = 4")
connection.execute(
    "SET preserve_insertion_order = false"
)

print("Connected to:", database_path)

Connected to: C:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\artifacts\practice_analytics.duckdb


## 3. Reusable notebook helpers

These functions execute SQL modules and display bounded table previews.

In [7]:
def execute_sql_file(file_name: str) -> None:
    """Read and execute one SQL module from sql/duckdb."""

    sql_path = sql_dir / file_name

    if not sql_path.exists():
        raise FileNotFoundError(
            f"SQL file does not exist: {sql_path}"
        )

    sql_text = sql_path.read_text(
        encoding="utf-8"
    )

    connection.execute(sql_text)

    print(f"Executed successfully: {file_name}")

In [8]:
def preview_table(
    table_name: str,
    row_limit: int = 10,
) -> pd.DataFrame:

    query = f"""
        SELECT *
        FROM {table_name}
        LIMIT {row_limit}
    """

    result = connection.execute(
        query
    ).fetchdf()

    return result

In [9]:
def describe_table(
    table_name: str,
) -> pd.DataFrame:

    result = connection.execute(
        f"DESCRIBE {table_name}"
    ).fetchdf()

    return result

## 4. Create the raw layer

Load exactly seven original CSV files into the `raw` schema.

`dim_sessions.csv` is not accepted as a source.

In [10]:
ingest_raw(
    connection,
    source_csv_dir,
)

print("Raw ingestion completed.")

Loading raw.users
Loading raw.products
Loading raw.orders
Loading raw.order_items
Loading raw.events


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loading raw.inventory_events
Loading raw.distribution_centers
Raw ingestion completed.


In [11]:
raw_objects = connection.execute(
    """
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = 'raw'
    ORDER BY table_name
    """
).fetchdf()

display(raw_objects)

,table_schema,table_name,table_type
0,raw,distribution_centers,BASE TABLE
1,raw,events,BASE TABLE
2,raw,inventory_events,BASE TABLE
3,raw,order_items,BASE TABLE
4,raw,orders,BASE TABLE
5,raw,products,BASE TABLE
6,raw,users,BASE TABLE


In [12]:
expected_raw_counts = {
    "users": 100_000,
    "products": 29_120,
    "orders": 124_814,
    "order_items": 180_862,
    "events": 2_420_661,
    "inventory_events": 488_146,
    "distribution_centers": 10,
}

raw_count_records = []

for table in source_tables:
    actual_rows = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM raw.{table}
        """
    ).fetchone()[0]

    expected_rows = expected_raw_counts[table]

    raw_count_records.append(
        {
            "table": f"raw.{table}",
            "expected_rows": expected_rows,
            "actual_rows": actual_rows,
            "status": (
                "PASS"
                if actual_rows == expected_rows
                else "FAIL"
            ),
        }
    )

raw_count_results = pd.DataFrame(
    raw_count_records
)

display(raw_count_results)

,table,expected_rows,actual_rows,status
0,raw.users,100000,100000,PASS
1,raw.products,29120,29120,PASS
2,raw.orders,124814,124814,PASS
3,raw.order_items,180862,180862,PASS
4,raw.events,2420661,2420661,PASS
5,raw.inventory_events,488146,488146,PASS
6,raw.distribution_centers,10,10,PASS


## 5. Build the staging layer

Execute `01_staging.sql`.

The staging layer:

- standardizes data types and names;
- handles blanks and missing text;
- excludes direct PII;
- derives sessions from events.

In [13]:
execute_sql_file(
    "01_staging.sql"
)

Executed successfully: 01_staging.sql


In [14]:
staging_objects = connection.execute(
    """
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = 'stg'
    ORDER BY table_name
    """
).fetchdf()

display(staging_objects)

,table_schema,table_name,table_type
0,stg,distribution_centers,VIEW
1,stg,events,VIEW
2,stg,inventory_events,VIEW
3,stg,order_items,VIEW
4,stg,orders,VIEW
5,stg,products,VIEW
6,stg,sessions,VIEW
7,stg,users,VIEW


### Inspect cleaned users

In [15]:
describe_table(
    "stg.users"
)

,column_name,column_type,null,key,default,extra
0,user_id,BIGINT,YES,None,None,None
1,age,INTEGER,YES,None,None,None
2,gender,VARCHAR,YES,None,None,None
3,state,VARCHAR,YES,None,None,None
4,postal_code,VARCHAR,YES,None,None,None
5,city,VARCHAR,YES,None,None,None
6,country,VARCHAR,YES,None,None,None
7,latitude,DOUBLE,YES,None,None,None
8,longitude,DOUBLE,YES,None,None,None
9,acquisition_source,VARCHAR,YES,None,None,None


In [16]:
preview_table(
    "stg.users",
    row_limit=10,
)

,user_id,age,gender,state,postal_code,city,country,latitude,longitude,acquisition_source,registered_at
0,65160,45,F,Yunnan,655099,Shenyang,China,25.476169,103.821625,Search,2024-03-02 02:10:00
1,66503,57,M,Yunnan,655099,Shenyang,China,25.476169,103.821625,Organic,2022-12-19 03:33:00
2,74713,24,M,Yunnan,655099,Shenyang,China,25.476169,103.821625,Search,2023-11-08 14:06:00
3,84699,15,M,Yunnan,655099,Shenyang,China,25.476169,103.821625,Search,2020-11-01 06:08:00
4,94002,27,F,Yunnan,655099,Shenyang,China,25.476169,103.821625,Search,2020-07-04 09:37:00
5,98350,19,F,Yunnan,655099,Shenyang,China,25.476169,103.821625,Organic,2019-12-10 13:43:00
6,2979,35,F,Yunnan,655400,Xi'an,China,26.195020,104.125174,Display,2021-03-26 17:05:00
7,4964,67,M,Yunnan,655400,Xi'an,China,26.195020,104.125174,Search,2021-02-08 06:36:00
8,6154,14,M,Yunnan,655400,Xi'an,China,26.195020,104.125174,Search,2020-03-07 12:27:00
9,6461,16,F,Yunnan,655400,Xi'an,China,26.195020,104.125174,Search,2022-08-20 00:11:00


In [17]:
preview_table(
    "stg.orders",
    row_limit=10,
)

,order_id,user_id,order_status,gender,created_at,returned_at,shipped_at,delivered_at,item_count
0,71053,56711,Returned,M,2023-12-07 01:42:00,2023-12-10 18:38:00,2023-12-08 23:36:00,2023-12-09 14:52:00,4
1,71070,56725,Returned,M,2023-11-29 08:58:00,2023-12-04 21:28:00,2023-12-02 08:02:00,2023-12-03 09:28:00,2
2,71089,56740,Returned,M,2022-06-05 00:48:00,2022-06-13 11:31:00,2022-06-06 10:31:00,2022-06-11 00:57:00,1
3,71116,56770,Returned,M,2021-10-27 03:35:00,2021-11-04 11:03:00,2021-10-29 16:26:00,2021-11-01 15:46:00,1
4,71119,56773,Returned,M,2021-09-14 09:07:00,2021-09-17 08:03:00,2021-09-14 22:54:00,2021-09-16 14:06:00,1
5,71121,56774,Returned,M,2020-07-13 03:19:00,2020-07-15 03:33:00,2020-07-13 03:22:00,2020-07-14 14:39:00,1
6,71124,56776,Returned,M,2023-11-26 17:34:00,2023-12-03 06:57:00,2023-11-28 07:44:00,2023-12-01 04:23:00,4
7,71130,56781,Returned,M,2024-05-05 00:09:00,2024-05-09 00:30:00,2024-05-06 00:45:00,2024-05-07 22:46:00,1
8,71159,56798,Returned,M,2023-05-18 02:42:00,2023-05-23 16:34:00,2023-05-18 19:57:00,2023-05-21 22:04:00,2
9,71168,56805,Returned,M,2023-06-23 06:23:00,2023-06-29 21:16:00,2023-06-23 18:22:00,2023-06-27 19:29:00,1


In [18]:
preview_table(
    "stg.order_items",
    row_limit=10,
)

,order_item_id,order_id,user_id,product_id,inventory_item_id,item_status,created_at,shipped_at,delivered_at,returned_at,sale_price
0,73961,50945,40676,29102,199711,Complete,2022-12-06 03:05:00,2022-12-07 21:25:00,2022-12-10 05:33:00,NaT,13.95
1,75007,51644,41211,18211,202541,Complete,2024-01-13 15:15:43,2024-01-12 22:07:00,2024-01-17 14:12:00,NaT,13.95
2,80207,55241,44087,18211,216520,Complete,2023-06-08 01:23:24,2023-06-09 11:49:00,2023-06-13 11:58:00,NaT,13.95
3,90428,62275,49663,6037,244164,Complete,2024-01-16 23:35:10,2024-01-16 10:14:00,2024-01-18 05:48:00,NaT,13.95
4,95760,65966,52601,29102,258602,Complete,2021-07-13 02:39:06,2021-07-11 08:30:00,2021-07-12 18:36:00,NaT,13.95
5,97758,67322,53708,26192,263981,Complete,2023-08-09 13:29:39,2023-08-11 19:42:00,2023-08-14 13:47:00,NaT,13.95
6,97766,67328,53713,25018,264002,Complete,2023-05-30 13:51:54,2023-05-31 01:45:00,2023-06-03 19:44:00,NaT,13.95
7,124065,85542,68353,26192,334947,Complete,2023-12-28 00:37:16,2023-12-28 03:31:00,2024-01-01 00:56:00,NaT,13.95
8,126684,87329,69835,6210,341979,Complete,2022-07-14 06:25:30,2022-07-16 21:18:00,2022-07-17 01:56:00,NaT,13.95
9,138109,95293,76259,9176,372805,Complete,2023-11-28 03:43:27,2023-11-27 10:19:00,2023-11-29 03:22:00,NaT,13.95


In [19]:
preview_table(
    "stg.products",
    row_limit=10,
)

,product_id,unit_cost,category,product_name,brand,retail_price,department,sku,distribution_center_id
0,13842,2.51875,Accessories,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,EBD58B8A3F1D72F4206201DA62FB1204,1
1,13928,2.33835,Accessories,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,2EAC42424D12436BDD6A5B8A88480CC3,1
2,14115,4.87956,Accessories,Enzyme Regular Solid Army Caps-Black W35S45D,MG,10.99,Women,EE364229B2791D1EF9355708EFF0BA34,1
3,14157,4.64877,Accessories,Enzyme Regular Solid Army Caps-Olive W35S45D (...,MG,10.99,Women,00BD13095D06C20B11A2993CA419D16B,1
4,14273,6.50793,Accessories,Washed Canvas Ivy Cap - Black W11S64C,MG,15.99,Women,F531DC20FDE20B7ADF3A73F52B71D0AF,1
5,15674,3.10625,Plus,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,63894CE404B8C652915C41EF8B879D20,1
6,15816,3.17730,Plus,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,151EA8C2D98CE89C2336324C11B1E107,1
7,28646,8.73563,Accessories,4 Panel Large Bill Flap Hat W15S48B (One Size ...,MG,19.99,Men,789334DE6DAA80D83AB4ACB6A4BF5AC7,1
8,28670,2.67594,Accessories,Low Profile Dyed Cotton Twill Cap - Black W39S55D,MG,6.18,Men,E74843B99DA8B29775C6AA9080436844,1
9,28714,2.27500,Accessories,Low Profile Dyed Cotton Twill Cap - Khaki W39S55D,MG,6.25,Men,8CA33D44648CC9FEECEF21E5A7123291,1


In [20]:
preview_table(
    "stg.inventory_events",
    row_limit=10,
)

,inventory_item_id,product_id,created_at,sold_at,unit_cost,category,product_name,brand,retail_price,department,sku,distribution_center_id
0,347248,13130,2021-07-25 07:17:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
1,350839,13130,2021-06-06 04:19:30,2021-07-03 03:58:30,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
2,350840,13130,2023-05-27 17:52:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
3,350841,13130,2023-10-31 08:20:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
4,350842,13130,2022-04-11 13:46:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
5,471788,13130,2023-03-07 06:59:26,2023-04-06 04:34:26,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
6,471789,13130,2024-05-04 02:55:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
7,471790,13130,2021-11-27 06:46:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
8,54720,12954,2023-12-30 14:23:56,2024-01-31 08:32:56,35.811229,Swim,Miraclesuit Women's Scales Models Atlantis One...,Miraclesuit,94.989998,Women,AB96C631D50E81C5F961BCBF25E49475,5
9,54721,12954,2020-05-02 17:34:00,NaT,35.811229,Swim,Miraclesuit Women's Scales Models Atlantis One...,Miraclesuit,94.989998,Women,AB96C631D50E81C5F961BCBF25E49475,5


In [21]:
preview_table(
    "stg.distribution_centers",
    row_limit=10,
)

,distribution_center_id,distribution_center_name,latitude,longitude
0,1,Memphis TN,35.1174,-89.9711
1,2,Chicago IL,41.8369,-87.6847
2,3,Houston TX,29.7604,-95.3698
3,4,Los Angeles CA,34.0500,-118.2500
4,5,New Orleans LA,29.9500,-90.0667
5,6,Port Authority of New York/New Jersey NY/NJ,40.6340,-73.7834
6,7,Philadelphia PA,39.9500,-75.1667
7,8,Mobile AL,30.6944,-88.0431
8,9,Charleston SC,32.7833,-79.9333
9,10,Savannah GA,32.0167,-81.1167


In [22]:
preview_table(
    "stg.sessions",
    row_limit=10,
)

,session_id,user_id,event_count,events_with_user_id,event_identity_coverage_rate,session_start_at,session_end_at,city,state,postal_code,browser,traffic_source,first_event_type,final_event_type,home_flag,department_flag,product_view_flag,cart_flag,purchase_flag,cancel_event_flag
0,8d4ff0bd-4c4f-4e0b-8681-7cf31c94ea4e,<NA>,3,0,0.0,2020-11-08 00:50:00,2020-11-08 01:17:00,Shenyang,Henan,453007,Firefox,Email,product,cancel,0,0,1,1,0,1
1,bab14919-e09c-41ac-963b-96d8edd9a4d0,<NA>,3,0,0.0,2023-12-15 05:14:00,2023-12-15 05:39:00,Seongnam City,Gyeonggi-do,462-120,Firefox,Adwords,product,cancel,0,0,1,1,0,1
2,a7a6e115-8cff-44d2-9ef3-d2f41cfe2b46,<NA>,1,0,0.0,2022-04-17 05:02:00,2022-04-17 05:02:00,Seongnam City,Gyeonggi-do,462-121,Chrome,YouTube,product,product,0,0,1,0,0,0
3,87a4bb1c-4767-479b-bbcb-0891651c93b7,<NA>,2,0,0.0,2020-04-05 14:23:00,2020-04-05 14:41:00,Bottrop,Nordrhein-Westfalen,46244,Chrome,Adwords,department,product,0,1,1,0,0,0
4,07c6e5c1-b551-44bd-8920-08c5d5c44051,<NA>,3,0,0.0,2023-04-23 02:18:00,2023-04-23 02:52:00,Rhede,Nordrhein-Westfalen,46414,Other,Organic,department,cart,0,1,1,1,0,0
5,43a9fbb6-0d36-438e-b267-3a874e25c641,<NA>,3,0,0.0,2023-08-08 11:59:00,2023-08-08 12:43:00,Santa Maria da Vitória,Bahia,47640-000,Chrome,Email,product,cancel,0,0,1,1,0,1
6,f5a15720-ab68-460a-bbe9-186685f2ff92,18586,7,7,1.0,2023-02-23 04:18:25,2023-02-24 04:30:33,Crawfordsville,Indiana,47933,Firefox,Email,department,purchase,0,1,1,1,1,0
7,4cccd60c-3ee9-4e79-b298-a3a75dbffc80,31809,10,10,1.0,2023-10-30 13:32:17,2023-11-01 13:48:47,Uijeongbuxi,Gyeonggi-do,480-010,Safari,Facebook,department,purchase,0,1,1,1,1,0
8,9e66cafa-15c0-4cb2-9564-82e97518e38b,11666,13,13,1.0,2024-04-09 17:35:54,2024-04-12 17:54:43,Uijeongbuxi,Gyeonggi-do,480-020,Firefox,Email,department,purchase,0,1,1,1,1,0
9,d1f5dd25-a7d9-4000-a39d-5f2784881046,<NA>,3,0,0.0,2022-05-26 08:52:00,2022-05-26 09:22:00,Pojuca,Bahia,48120-000,Firefox,Email,product,cancel,0,0,1,1,0,1


### Inspect derived sessions

In [23]:
describe_table(
    "stg.sessions"
)

,column_name,column_type,null,key,default,extra
0,session_id,VARCHAR,YES,None,None,None
1,user_id,BIGINT,YES,None,None,None
2,event_count,INTEGER,YES,None,None,None
3,events_with_user_id,INTEGER,YES,None,None,None
4,event_identity_coverage_rate,DOUBLE,YES,None,None,None
5,session_start_at,TIMESTAMP,YES,None,None,None
6,session_end_at,TIMESTAMP,YES,None,None,None
7,city,VARCHAR,YES,None,None,None
8,state,VARCHAR,YES,None,None,None
9,postal_code,VARCHAR,YES,None,None,None


In [24]:
preview_table(
    "stg.sessions",
    row_limit=10,
)

,session_id,user_id,event_count,events_with_user_id,event_identity_coverage_rate,session_start_at,session_end_at,city,state,postal_code,browser,traffic_source,first_event_type,final_event_type,home_flag,department_flag,product_view_flag,cart_flag,purchase_flag,cancel_event_flag
0,a5de7fc3-56b6-4c08-8c2d-35f81da46ec5,<NA>,3,0,0.0,2021-08-02 15:09:00,2021-08-02 15:27:00,Zhuhai,Shandong,273500,Firefox,Adwords,department,cart,0,1,1,1,0,0
1,6326af5f-2703-4120-88f5-0498a25fff78,<NA>,2,0,0.0,2022-10-18 04:21:00,2022-10-18 04:41:00,Cary,North Carolina,27518,Chrome,Facebook,department,product,0,1,1,0,0,0
2,9a685298-23f4-4689-828b-9ead724135ef,54907,13,13,1.0,2023-02-25 08:41:13,2023-02-28 08:57:29,Ganderkesee,Niedersachsen,27777,Chrome,Organic,department,purchase,0,1,1,1,1,0
3,5db17cb0-1165-4d89-9028-5152169632f8,55343,5,5,1.0,2021-11-04 06:27:03,2021-11-04 06:34:51,Mooresville,North Carolina,28115,Chrome,Adwords,home,purchase,1,1,1,1,1,0
4,8ef1ef66-7c0e-474e-a424-9ee127e3a850,46940,13,13,1.0,2024-02-25 13:46:56,2024-02-25 14:01:49,North Charleston,South Carolina,29405,Chrome,Adwords,department,purchase,0,1,1,1,1,0
5,1513ae3c-d334-47b6-a664-a7b73efe14d3,<NA>,2,0,0.0,2022-11-07 09:01:00,2022-11-07 09:01:00,York,South Carolina,29745,Safari,Email,department,product,0,1,1,0,0,0
6,56a8adb0-f387-415a-8709-4b2e731e3467,18054,7,7,1.0,2021-04-12 23:43:37,2021-04-12 23:50:49,Tucker,Georgia,30087,Chrome,Facebook,department,purchase,0,1,1,1,1,0
7,a66e82dd-c8de-4575-bc7c-d60d99475100,<NA>,2,0,0.0,2023-05-29 00:26:00,2023-05-29 00:49:00,Melbourne,Victoria,3033,Chrome,Email,department,product,0,1,1,0,0,0
8,ecc4a251-c901-4261-9cbb-4b60bc4f80c4,13605,10,10,1.0,2022-07-24 02:52:19,2022-07-28 03:03:56,Dunwoody,Georgia,30338,Safari,Facebook,department,purchase,0,1,1,1,1,0
9,5d7941d4-e3c6-432a-bedd-41cda378e18c,50087,13,13,1.0,2022-01-14 23:12:16,2022-01-17 23:31:07,Hannover,Niedersachsen,30419,Safari,Organic,department,purchase,0,1,1,1,1,0


In [25]:
preview_table(
    "stg.inventory_events",
    row_limit=10,
)

,inventory_item_id,product_id,created_at,sold_at,unit_cost,category,product_name,brand,retail_price,department,sku,distribution_center_id
0,347248,13130,2021-07-25 07:17:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
1,350839,13130,2021-06-06 04:19:30,2021-07-03 03:58:30,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
2,350840,13130,2023-05-27 17:52:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
3,350841,13130,2023-10-31 08:20:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
4,350842,13130,2022-04-11 13:46:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
5,471788,13130,2023-03-07 06:59:26,2023-04-06 04:34:26,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
6,471789,13130,2024-05-04 02:55:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
7,471790,13130,2021-11-27 06:46:00,NaT,62.196000,Swim,Miraclesuit Women's Roller Coaster Escape One ...,Miraclesuit,146.000000,Women,1686C5EC96F728148F941AB2B0F2CC35,5
8,54720,12954,2023-12-30 14:23:56,2024-01-31 08:32:56,35.811229,Swim,Miraclesuit Women's Scales Models Atlantis One...,Miraclesuit,94.989998,Women,AB96C631D50E81C5F961BCBF25E49475,5
9,54721,12954,2020-05-02 17:34:00,NaT,35.811229,Swim,Miraclesuit Women's Scales Models Atlantis One...,Miraclesuit,94.989998,Women,AB96C631D50E81C5F961BCBF25E49475,5


In [26]:
#stagign row counts
staging_tables = [
    "users",
    "products",
    "orders",
    "order_items",
    "events",
    "sessions",
    "inventory_events",
    "distribution_centers",
]

staging_count_records = []

for table in staging_tables:
    rows = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM stg.{table}
        """
    ).fetchone()[0]

    staging_count_records.append(
        {
            "table": f"stg.{table}",
            "rows": rows,
        }
    )

staging_count_results = pd.DataFrame(
    staging_count_records
)

display(staging_count_results)

,table,rows
0,stg.users,100000
1,stg.products,29120
2,stg.orders,124814
3,stg.order_items,180862
4,stg.events,2420661
5,stg.sessions,680862
6,stg.inventory_events,488146
7,stg.distribution_centers,10


### Validate event-to-session reconciliation

Confirm that:

- every distinct usable session ID produces one session;
- no sessionizable events are lost;
- no sessionizable events are duplicated.

In [27]:
session_reconciliation = connection.execute(
    """
    SELECT
        (
            SELECT COUNT(*)
            FROM stg.events
        ) AS all_events,

        (
            SELECT COUNT(*)
            FROM stg.events
            WHERE session_id IS NOT NULL
        ) AS sessionizable_events,

        (
            SELECT COUNT(DISTINCT session_id)
            FROM stg.events
            WHERE session_id IS NOT NULL
        ) AS distinct_session_ids,

        (
            SELECT COUNT(*)
            FROM stg.sessions
        ) AS derived_sessions,

        (
            SELECT SUM(event_count)
            FROM stg.sessions
        ) AS modeled_events
    """
).fetchdf()

session_reconciliation[
    "session_count_status"
] = session_reconciliation.apply(
    lambda row: (
        "PASS"
        if row["distinct_session_ids"]
           == row["derived_sessions"]
        else "FAIL"
    ),
    axis=1,
)

session_reconciliation[
    "event_count_status"
] = session_reconciliation.apply(
    lambda row: (
        "PASS"
        if row["sessionizable_events"]
           == row["modeled_events"]
        else "FAIL"
    ),
    axis=1,
)

display(session_reconciliation)

,all_events,sessionizable_events,distinct_session_ids,derived_sessions,modeled_events,session_count_status,event_count_status
0,2420661,2420661,680862,680862,2420661.0,PASS,PASS


In [28]:
#verify direct PII exclusion
direct_pii_columns = [
    "first_name",
    "last_name",
    "email",
    "street_address",
    "ip_address",
]

pii_check = connection.execute(
    """
    SELECT
        table_schema,
        table_name,
        column_name
    FROM information_schema.columns
    WHERE table_schema = 'stg'
      AND column_name IN (
          'first_name',
          'last_name',
          'email',
          'street_address',
          'ip_address'
      )
    ORDER BY
        table_name,
        column_name
    """
).fetchdf()

display(pii_check)

if pii_check.empty:
    print(
        "PASS: direct PII is absent "
        "from staging views."
    )
else:
    print(
        "REVIEW: direct PII columns "
        "were found in staging."
    )

,table_schema,table_name,column_name


PASS: direct PII is absent from staging views.


## 6. Build core dimensions

Execute `02_core_dimensions.sql`.

Core dimensions provide reusable descriptive entities for facts, marts, and Power BI.

In [29]:
execute_sql_file(
    "02_core_dimensions.sql"
)

Executed successfully: 02_core_dimensions.sql


In [30]:
#list dimensions
dimension_objects = connection.execute(
    """
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = 'core'
      AND table_name LIKE 'dim_%'
    ORDER BY table_name
    """
).fetchdf()

display(dimension_objects)

,table_schema,table_name,table_type
0,core,dim_acquisition_source,BASE TABLE
1,core,dim_customer,BASE TABLE
2,core,dim_date,BASE TABLE
3,core,dim_distribution_center,BASE TABLE
4,core,dim_product,BASE TABLE
5,core,dim_session_traffic_source,BASE TABLE


In [31]:
#inspect every dimension
dimension_tables = [
    "core.dim_date",
    "core.dim_customer",
    "core.dim_product",
    "core.dim_distribution_center",
    "core.dim_session_traffic_source",
    "core.dim_acquisition_source",
]

for table in dimension_tables:
    print(f"\n--- {table} ---")

    preview_table(
        table,
        row_limit=5,
    )


--- core.dim_date ---

--- core.dim_customer ---

--- core.dim_product ---

--- core.dim_distribution_center ---

--- core.dim_session_traffic_source ---

--- core.dim_acquisition_source ---


## 7. Dimension validation

Compare source and output row counts, confirm unique keys, and validate registration-date relationships.

In [32]:
dimension_tests = connection.execute(
    """
    WITH tests AS (

        SELECT
            'Customer rows preserved'
                AS test_name,

            (SELECT COUNT(*) FROM stg.users)
                AS expected_value,

            (SELECT COUNT(*)
             FROM core.dim_customer)
                AS actual_value

        UNION ALL

        SELECT
            'Customer key is unique',

            (SELECT COUNT(*)
             FROM core.dim_customer),

            (SELECT COUNT(DISTINCT user_id)
             FROM core.dim_customer)

        UNION ALL

        SELECT
            'Product rows preserved',

            (SELECT COUNT(*)
             FROM stg.products),

            (SELECT COUNT(*)
             FROM core.dim_product)

        UNION ALL

        SELECT
            'Product key is unique',

            (SELECT COUNT(*)
             FROM core.dim_product),

            (SELECT COUNT(DISTINCT product_id)
             FROM core.dim_product)

        UNION ALL

        SELECT
            'Distribution centers preserved',

            (SELECT COUNT(*)
             FROM stg.distribution_centers),

            (SELECT COUNT(*)
             FROM core.dim_distribution_center)

        UNION ALL

        SELECT
            'Distribution-center key is unique',

            (SELECT COUNT(*)
             FROM core.dim_distribution_center),

            (
                SELECT COUNT(
                    DISTINCT distribution_center_id
                )
                FROM core.dim_distribution_center
            )

        UNION ALL

        SELECT
            'Session sources preserved',

            (
                SELECT COUNT(
                    DISTINCT traffic_source
                )
                FROM stg.sessions
            ),

            (
                SELECT COUNT(*)
                FROM core.dim_session_traffic_source
            )

        UNION ALL

        SELECT
            'Session-source key is unique',

            (
                SELECT COUNT(*)
                FROM core.dim_session_traffic_source
            ),

            (
                SELECT COUNT(
                    DISTINCT session_traffic_source_key
                )
                FROM core.dim_session_traffic_source
            )

        UNION ALL

        SELECT
            'Acquisition sources preserved',

            (
                SELECT COUNT(
                    DISTINCT acquisition_source
                )
                FROM stg.users
            ),

            (
                SELECT COUNT(*)
                FROM core.dim_acquisition_source
            )

        UNION ALL

        SELECT
            'Acquisition-source key is unique',

            (
                SELECT COUNT(*)
                FROM core.dim_acquisition_source
            ),

            (
                SELECT COUNT(
                    DISTINCT acquisition_source_key
                )
                FROM core.dim_acquisition_source
            )

        UNION ALL

        SELECT
            'Date key is unique',

            (SELECT COUNT(*)
             FROM core.dim_date),

            (SELECT COUNT(DISTINCT date_key)
             FROM core.dim_date)

        UNION ALL

        SELECT
            'Registration date keys matched',

            (
                SELECT COUNT(*)
                FROM core.dim_customer
                WHERE registered_date_key IS NOT NULL
            ),

            (
                SELECT COUNT(*)
                FROM core.dim_customer AS customer

                INNER JOIN core.dim_date AS calendar
                    ON customer.registered_date_key
                       = calendar.date_key

                WHERE customer.registered_date_key
                      IS NOT NULL
            )
    )

    SELECT
        test_name,
        expected_value,
        actual_value,

        CASE
            WHEN expected_value = actual_value
                THEN 'PASS'
            ELSE 'FAIL'
        END AS test_result

    FROM tests

    ORDER BY test_name
    """
).fetchdf()

display(dimension_tests)

,test_name,expected_value,actual_value,test_result
0,Acquisition sources preserved,5,5,PASS
1,Acquisition-source key is unique,5,5,PASS
2,Customer key is unique,100000,100000,PASS
3,Customer rows preserved,100000,100000,PASS
4,Date key is unique,1994,1994,PASS
5,Distribution centers preserved,10,10,PASS
6,Distribution-center key is unique,10,10,PASS
7,Product key is unique,29120,29120,PASS
8,Product rows preserved,29120,29120,PASS
9,Registration date keys matched,100000,100000,PASS


In [33]:
#Product price bands
price_band_summary = connection.execute(
    """
    SELECT
        price_band,
        COUNT(*) AS products,
        MIN(retail_price) AS minimum_price,
        MAX(retail_price) AS maximum_price
    FROM core.dim_product
    GROUP BY price_band
    ORDER BY minimum_price
    """
).fetchdf()

display(price_band_summary)

,price_band,products,minimum_price,maximum_price
0,Under $25,7839,0.02,24.990000
1,$25-$49,10074,25.00,49.990002
2,$50-$99,7148,50.00,99.989998
3,$100-$199,3213,100.00,199.990005
4,$200+,846,200.00,999.000000


In [34]:
#Date range
date_dimension_summary = connection.execute(
    """
    SELECT
        MIN(calendar_date) AS first_date,
        MAX(calendar_date) AS last_date,
        COUNT(*) AS calendar_days,
        SUM(is_weekend) AS weekend_days,
        SUM(is_complete_month) AS complete_month_days
    FROM core.dim_date
    """
).fetchdf()

display(date_dimension_summary)

,first_date,last_date,calendar_days,weekend_days,complete_month_days
0,2018-11-28,2024-05-13,1994,570.0,1981.0


In [35]:
#Product margins
product_margin_summary = connection.execute(
    """
    SELECT
        MIN(unit_margin) AS minimum_margin,
        MAX(unit_margin) AS maximum_margin,
        AVG(unit_margin) AS average_margin,

        SUM(
            CASE
                WHEN unit_margin < 0 THEN 1
                ELSE 0
            END
        ) AS negative_margin_products

    FROM core.dim_product
    """
).fetchdf()

display(product_margin_summary)

,minimum_margin,maximum_margin,average_margin,negative_margin_products
0,0.0117,594.404999,30.73839,0.0


In [36]:
for table in dimension_tables:
    print(f"\n--- {table} ---")

    display(
        preview_table(
            table,
            row_limit=5,
        )
    )


--- core.dim_date ---


,date_key,calendar_date,year,quarter_number,quarter_label,month_number,month_name,month_year,month_start,quarter_start,year_start,iso_week_number,iso_day_of_week,day_name,is_weekend,is_complete_month
0,20240513,2024-05-13,2024,2,Q2,5,May,May 2024,2024-05-01,2024-04-01,2024-01-01,20,1,Monday,0,0
1,20240512,2024-05-12,2024,2,Q2,5,May,May 2024,2024-05-01,2024-04-01,2024-01-01,19,7,Sunday,1,0
2,20240511,2024-05-11,2024,2,Q2,5,May,May 2024,2024-05-01,2024-04-01,2024-01-01,19,6,Saturday,1,0
3,20240510,2024-05-10,2024,2,Q2,5,May,May 2024,2024-05-01,2024-04-01,2024-01-01,19,5,Friday,0,0
4,20240509,2024-05-09,2024,2,Q2,5,May,May 2024,2024-05-01,2024-04-01,2024-01-01,19,4,Thursday,0,0



--- core.dim_customer ---


,user_id,age,age_band,gender,country,state,city,postal_code,latitude,longitude,acquisition_source,registered_at,registered_date_key
0,65160,45,45-54,F,China,Yunnan,Shenyang,655099,25.476169,103.821625,Search,2024-03-02 02:10:00,20240302
1,66503,57,55-64,M,China,Yunnan,Shenyang,655099,25.476169,103.821625,Organic,2022-12-19 03:33:00,20221219
2,74713,24,18-24,M,China,Yunnan,Shenyang,655099,25.476169,103.821625,Search,2023-11-08 14:06:00,20231108
3,84699,15,18-24,M,China,Yunnan,Shenyang,655099,25.476169,103.821625,Search,2020-11-01 06:08:00,20201101
4,94002,27,25-34,F,China,Yunnan,Shenyang,655099,25.476169,103.821625,Search,2020-07-04 09:37:00,20200704



--- core.dim_product ---


...

,distribution_center_id,distribution_center_name,latitude,longitude
0,1,Memphis TN,35.1174,-89.9711
1,2,Chicago IL,41.8369,-87.6847
2,3,Houston TX,29.7604,-95.3698
3,4,Los Angeles CA,34.0500,-118.2500
4,5,New Orleans LA,29.9500,-90.0667



--- core.dim_session_traffic_source ---


,session_traffic_source_key,traffic_source
0,1,Adwords
1,2,Email
2,3,Facebook
3,4,Organic
4,5,YouTube



--- core.dim_acquisition_source ---


,acquisition_source_key,acquisition_source
0,1,Display
1,2,Email
2,3,Facebook
3,4,Organic
4,5,Search


## 8. Build core facts

Execute `03_core_facts.sql`.

Fact-table grains:

- `core.fact_order`: one row per order;
- `core.fact_order_item`: one row per product line in an order;
- `core.fact_session`: one row per website session;
- `core.fact_inventory`: one row per physical inventory item.

In [37]:
execute_sql_file(
    "03_core_facts.sql"
)

Executed successfully: 03_core_facts.sql


In [38]:
#List created fact tables
fact_objects = connection.execute(
    """
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = 'core'
      AND table_name LIKE 'fact_%'
    ORDER BY table_name
    """
).fetchdf()

display(fact_objects)

,table_schema,table_name,table_type
0,core,fact_inventory,BASE TABLE
1,core,fact_order,BASE TABLE
2,core,fact_order_item,BASE TABLE
3,core,fact_session,BASE TABLE


In [39]:
#Define the fact-table list
fact_tables = [
    "core.fact_order",
    "core.fact_order_item",
    "core.fact_session",
    "core.fact_inventory",
]

In [40]:
#Inspect structures and sample rows
for table in fact_tables:
    print(f"\n{'=' * 70}")
    print(table)
    print(f"{'=' * 70}")

    print("\nStructure:")
    display(
        describe_table(table)
    )

    print("\nSample rows:")
    display(
        preview_table(
            table,
            row_limit=5,
        )
    )


core.fact_order

Structure:


,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,customer_id,BIGINT,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_created_at,TIMESTAMP,YES,None,None,None
4,shipped_at,TIMESTAMP,YES,None,None,None
5,delivered_at,TIMESTAMP,YES,None,None,None
6,returned_at,TIMESTAMP,YES,None,None,None
7,order_date_key,INTEGER,YES,None,None,None
8,item_count,INTEGER,YES,None,None,None
9,acquisition_source,VARCHAR,YES,None,None,None



Sample rows:


,order_id,customer_id,order_status,order_created_at,shipped_at,delivered_at,returned_at,order_date_key,item_count,acquisition_source,ship_lead_days,delivery_lead_days,end_to_end_lead_days,return_cycle_days,valid_timeline_flag,cancelled_order_flag,returned_order_flag,completed_order_flag
0,71053,56711,Returned,2023-12-07 01:42:00,2023-12-08 23:36:00,2023-12-09 14:52:00,2023-12-10 18:38:00,20231207,4,Organic,1.912500,0.636111,2.548611,1.156944,1,0,1,0
1,71070,56725,Returned,2023-11-29 08:58:00,2023-12-02 08:02:00,2023-12-03 09:28:00,2023-12-04 21:28:00,20231129,2,Search,2.961111,1.059722,4.020833,1.500000,1,0,1,0
2,71089,56740,Returned,2022-06-05 00:48:00,2022-06-06 10:31:00,2022-06-11 00:57:00,2022-06-13 11:31:00,20220605,1,Organic,1.404861,4.601389,6.006250,2.440278,1,0,1,0
3,71116,56770,Returned,2021-10-27 03:35:00,2021-10-29 16:26:00,2021-11-01 15:46:00,2021-11-04 11:03:00,20211027,1,Search,2.535417,2.972222,5.507639,2.803472,1,0,1,0
4,71119,56773,Returned,2021-09-14 09:07:00,2021-09-14 22:54:00,2021-09-16 14:06:00,2021-09-17 08:03:00,20210914,1,Display,0.574306,1.633333,2.207639,0.747917,1,0,1,0



core.fact_order_item

Structure:


...

,session_id,customer_id,session_start_at,session_end_at,session_date_key,browser,traffic_source,country,event_count,events_with_user_id,event_identity_coverage_rate,identified_session_flag,session_duration_seconds,first_event_type,final_event_type,home_flag,department_flag,product_view_flag,cart_flag,purchase_flag,cancel_event_flag,cart_abandoned_flag,early_product_exit_flag,highest_stage
0,590f8cc8-53bd-4cf8-8776-ea3731810d85,<NA>,2023-03-23 05:34:00,2023-03-23 05:39:00,20230323,Firefox,Adwords,Unknown,2,0,0.0,0,300,department,product,0,1,1,0,0,0,0,1,Product
1,8255891c-e148-447f-99e1-048d3b7adc22,<NA>,2020-02-21 07:06:00,2020-02-21 07:28:00,20200221,Firefox,Adwords,Unknown,3,0,0.0,0,1320,department,cart,0,1,1,1,0,0,1,0,Cart
2,ad757e4f-eb6a-4527-860f-6a1fc04ebd95,<NA>,2020-09-08 02:53:00,2020-09-08 03:20:00,20200908,Chrome,Email,Unknown,2,0,0.0,0,1620,department,product,0,1,1,0,0,0,0,1,Product
3,d2f171cd-7138-4d5e-962a-e8d183808230,<NA>,2020-09-18 11:48:00,2020-09-18 11:53:00,20200918,Safari,Organic,Unknown,2,0,0.0,0,300,department,product,0,1,1,0,0,0,0,1,Product
4,ead921c8-1725-4419-99ef-78f26e10da46,<NA>,2022-05-03 09:02:00,2022-05-03 09:27:00,20220503,Chrome,Email,Unknown,3,0,0.0,0,1500,department,cart,0,1,1,1,0,0,1,0,Cart



core.fact_inventory

Structure:


,column_name,column_type,null,key,default,extra
0,inventory_item_id,BIGINT,YES,None,None,None
1,product_id,BIGINT,YES,None,None,None
2,distribution_center_id,INTEGER,YES,None,None,None
3,inventory_created_at,TIMESTAMP,YES,None,None,None
4,sold_at,TIMESTAMP,YES,None,None,None
5,inventory_created_date_key,INTEGER,YES,None,None,None
6,unit_cost,DOUBLE,YES,None,None,None
7,retail_price,DOUBLE,YES,None,None,None
8,sold_flag,INTEGER,YES,None,None,None
9,days_to_sell,BIGINT,YES,None,None,None



Sample rows:


,inventory_item_id,product_id,distribution_center_id,inventory_created_at,sold_at,inventory_created_date_key,unit_cost,retail_price,sold_flag,days_to_sell,unsold_age_days,inventory_age_bucket
0,373483,2095,8,2020-10-30 12:17:00,NaT,20201030,17.19212,38.119999,0,<NA>,1291,181+ days
1,401482,2095,8,2023-01-07 03:15:00,2023-02-07 10:05:00,20230107,17.19212,38.119999,1,31,<NA>,Sold
2,401483,2095,8,2024-01-18 00:57:00,NaT,20240118,17.19212,38.119999,0,<NA>,116,91-180 days
3,476042,2095,8,2023-11-12 01:38:54,2023-11-18 09:21:54,20231112,17.19212,38.119999,1,6,<NA>,Sold
4,476043,2095,8,2023-08-31 08:18:00,NaT,20230831,17.19212,38.119999,0,<NA>,256,181+ days


In [41]:
#Validate grains and row preservation
fact_grain_tests = connection.execute(
    """
    WITH tests AS (

        SELECT
            'Order rows preserved'
                AS test_name,

            (SELECT COUNT(*)
             FROM stg.orders)
                AS expected_value,

            (SELECT COUNT(*)
             FROM core.fact_order)
                AS actual_value

        UNION ALL

        SELECT
            'Order key is unique',

            (SELECT COUNT(*)
             FROM core.fact_order),

            (SELECT COUNT(DISTINCT order_id)
             FROM core.fact_order)

        UNION ALL

        SELECT
            'Order-item rows preserved',

            (SELECT COUNT(*)
             FROM stg.order_items),

            (SELECT COUNT(*)
             FROM core.fact_order_item)

        UNION ALL

        SELECT
            'Order-item key is unique',

            (SELECT COUNT(*)
             FROM core.fact_order_item),

            (SELECT COUNT(DISTINCT order_item_id)
             FROM core.fact_order_item)

        UNION ALL

        SELECT
            'Session rows preserved',

            (SELECT COUNT(*)
             FROM stg.sessions),

            (SELECT COUNT(*)
             FROM core.fact_session)

        UNION ALL

        SELECT
            'Session key is unique',

            (SELECT COUNT(*)
             FROM core.fact_session),

            (SELECT COUNT(DISTINCT session_id)
             FROM core.fact_session)

        UNION ALL

        SELECT
            'Inventory rows preserved',

            (SELECT COUNT(*)
             FROM stg.inventory_events),

            (SELECT COUNT(*)
             FROM core.fact_inventory)

        UNION ALL

        SELECT
            'Inventory key is unique',

            (SELECT COUNT(*)
             FROM core.fact_inventory),

            (SELECT COUNT(DISTINCT inventory_item_id)
             FROM core.fact_inventory)
    )

    SELECT
        test_name,
        expected_value,
        actual_value,

        CASE
            WHEN expected_value = actual_value
                THEN 'PASS'
            ELSE 'FAIL'
        END AS test_result

    FROM tests

    ORDER BY test_name
    """
).fetchdf()

display(fact_grain_tests)

,test_name,expected_value,actual_value,test_result
0,Inventory key is unique,488146,488146,PASS
1,Inventory rows preserved,488146,488146,PASS
2,Order key is unique,124814,124814,PASS
3,Order rows preserved,124814,124814,PASS
4,Order-item key is unique,180862,180862,PASS
5,Order-item rows preserved,180862,180862,PASS
6,Session key is unique,680862,680862,PASS
7,Session rows preserved,680862,680862,PASS


In [42]:
#Stop if a grain test fails
failed_fact_grain_tests = fact_grain_tests[
    fact_grain_tests["test_result"] != "PASS"
]

if failed_fact_grain_tests.empty:
    print(
        "PASS: all fact-table grain "
        "and row-preservation tests passed."
    )
else:
    display(failed_fact_grain_tests)

    raise AssertionError(
        "One or more fact-table grain tests failed."
    )

PASS: all fact-table grain and row-preservation tests passed.


In [43]:
#Check dimension relationships
fact_relationship_tests = connection.execute(
    """
    WITH tests AS (

        SELECT
            'Order facts without customer'
                AS test_name,

            COUNT(*) AS orphan_rows

        FROM core.fact_order AS fact

        LEFT JOIN core.dim_customer AS customer
            ON fact.customer_id = customer.user_id

        WHERE fact.customer_id IS NOT NULL
          AND customer.user_id IS NULL

        UNION ALL

        SELECT
            'Order-item facts without customer',

            COUNT(*)

        FROM core.fact_order_item AS fact

        LEFT JOIN core.dim_customer AS customer
            ON fact.customer_id = customer.user_id

        WHERE fact.customer_id IS NOT NULL
          AND customer.user_id IS NULL

        UNION ALL

        SELECT
            'Order-item facts without product',

            COUNT(*)

        FROM core.fact_order_item AS fact

        LEFT JOIN core.dim_product AS product
            ON fact.product_id = product.product_id

        WHERE fact.product_id IS NOT NULL
          AND product.product_id IS NULL

        UNION ALL

        SELECT
            'Session facts without known customer',

            COUNT(*)

        FROM core.fact_session AS fact

        LEFT JOIN core.dim_customer AS customer
            ON fact.customer_id = customer.user_id

        WHERE fact.customer_id IS NOT NULL
          AND customer.user_id IS NULL

        UNION ALL

        SELECT
            'Inventory facts without product',

            COUNT(*)

        FROM core.fact_inventory AS fact

        LEFT JOIN core.dim_product AS product
            ON fact.product_id = product.product_id

        WHERE fact.product_id IS NOT NULL
          AND product.product_id IS NULL

        UNION ALL

        SELECT
            'Inventory facts without distribution center',

            COUNT(*)

        FROM core.fact_inventory AS fact

        LEFT JOIN core.dim_distribution_center AS center
            ON fact.distribution_center_id
               = center.distribution_center_id

        WHERE fact.distribution_center_id IS NOT NULL
          AND center.distribution_center_id IS NULL
    )

    SELECT
        test_name,
        orphan_rows,

        CASE
            WHEN orphan_rows = 0
                THEN 'PASS'
            ELSE 'FAIL'
        END AS test_result

    FROM tests

    ORDER BY test_name
    """
).fetchdf()

display(fact_relationship_tests)

,test_name,orphan_rows,test_result
0,Inventory facts without distribution center,0,PASS
1,Inventory facts without product,0,PASS
2,Order facts without customer,0,PASS
3,Order-item facts without customer,0,PASS
4,Order-item facts without product,0,PASS
5,Session facts without known customer,0,PASS


In [44]:
#Validate fact date keys
fact_date_tests = connection.execute(
    """
    WITH tests AS (

        SELECT
            'Order date keys matched'
                AS test_name,

            COUNT(*) AS expected_rows,

            COUNT(calendar.date_key) AS matched_rows

        FROM core.fact_order AS fact

        LEFT JOIN core.dim_date AS calendar
            ON fact.order_date_key
               = calendar.date_key

        WHERE fact.order_date_key IS NOT NULL

        UNION ALL

        SELECT
            'Order-item date keys matched',

            COUNT(*),

            COUNT(calendar.date_key)

        FROM core.fact_order_item AS fact

        LEFT JOIN core.dim_date AS calendar
            ON fact.order_date_key
               = calendar.date_key

        WHERE fact.order_date_key IS NOT NULL

        UNION ALL

        SELECT
            'Session date keys matched',

            COUNT(*),

            COUNT(calendar.date_key)

        FROM core.fact_session AS fact

        LEFT JOIN core.dim_date AS calendar
            ON fact.session_date_key
               = calendar.date_key

        WHERE fact.session_date_key IS NOT NULL

        UNION ALL

        SELECT
            'Inventory date keys matched',

            COUNT(*),

            COUNT(calendar.date_key)

        FROM core.fact_inventory AS fact

        LEFT JOIN core.dim_date AS calendar
            ON fact.inventory_created_date_key
               = calendar.date_key

        WHERE fact.inventory_created_date_key IS NOT NULL
    )

    SELECT
        test_name,
        expected_rows,
        matched_rows,

        CASE
            WHEN expected_rows = matched_rows
                THEN 'PASS'
            ELSE 'FAIL'
        END AS test_result

    FROM tests

    ORDER BY test_name
    """
).fetchdf()

display(fact_date_tests)

,test_name,expected_rows,matched_rows,test_result
0,Inventory date keys matched,488146,488146,PASS
1,Order date keys matched,124814,124814,PASS
2,Order-item date keys matched,180862,180862,PASS
3,Session date keys matched,680862,680862,PASS


In [45]:
#Review timeline anomalies
timeline_quality = connection.execute(
    """
    SELECT
        'Orders' AS fact_name,
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN valid_timeline_flag = 0
                THEN 1
                ELSE 0
            END
        ) AS invalid_timeline_rows,

        ROUND(
            100.0
            * SUM(
                CASE
                    WHEN valid_timeline_flag = 0
                    THEN 1
                    ELSE 0
                END
            )
            / NULLIF(COUNT(*), 0),
            2
        ) AS invalid_timeline_percentage

    FROM core.fact_order

    UNION ALL

    SELECT
        'Order items',
        COUNT(*),

        SUM(
            CASE
                WHEN valid_timeline_flag = 0
                THEN 1
                ELSE 0
            END
        ),

        ROUND(
            100.0
            * SUM(
                CASE
                    WHEN valid_timeline_flag = 0
                    THEN 1
                    ELSE 0
                END
            )
            / NULLIF(COUNT(*), 0),
            2
        )

    FROM core.fact_order_item
    """
).fetchdf()

display(timeline_quality)

,fact_name,total_rows,invalid_timeline_rows,invalid_timeline_percentage
0,Orders,124814,0.0,0.0
1,Order items,180862,35440.0,19.6


In [46]:
#Session funnel summary
session_funnel_summary = connection.execute(
    """
    SELECT
        COUNT(*) AS sessions,

        SUM(home_flag) AS home_sessions,

        SUM(department_flag)
            AS department_sessions,

        SUM(product_view_flag)
            AS product_view_sessions,

        SUM(cart_flag)
            AS cart_sessions,

        SUM(purchase_flag)
            AS purchase_sessions,

        SUM(cart_abandoned_flag)
            AS cart_abandoned_sessions,

        SUM(early_product_exit_flag)
            AS early_product_exit_sessions,

        ROUND(
            100.0 * SUM(purchase_flag)
            / NULLIF(COUNT(*), 0),
            2
        ) AS session_purchase_rate,

        ROUND(
            100.0 * SUM(cart_abandoned_flag)
            / NULLIF(SUM(cart_flag), 0),
            2
        ) AS cart_abandonment_rate

    FROM core.fact_session
    """
).fetchdf()

display(session_funnel_summary)

,sessions,home_sessions,department_sessions,product_view_sessions,cart_sessions,purchase_sessions,cart_abandoned_sessions,early_product_exit_sessions,session_purchase_rate,cart_abandonment_rate
0,680862,87406.0,430836.0,680862.0,430614.0,180862.0,249752.0,250248.0,26.56,58.0


In [47]:
#Inventory summary
inventory_summary = connection.execute(
    """
    SELECT
        inventory_age_bucket,

        COUNT(*) AS inventory_units,

        SUM(sold_flag) AS sold_units,

        COUNT(*) - SUM(sold_flag)
            AS unsold_units,

        ROUND(
            AVG(days_to_sell),
            2
        ) AS average_days_to_sell,

        ROUND(
            SUM(unit_cost),
            2
        ) AS inventory_cost_value,

        ROUND(
            SUM(retail_price),
            2
        ) AS inventory_retail_value

    FROM core.fact_inventory

    GROUP BY inventory_age_bucket

    ORDER BY
        CASE inventory_age_bucket
            WHEN 'Sold' THEN 1
            WHEN '0-30 days' THEN 2
            WHEN '31-60 days' THEN 3
            WHEN '61-90 days' THEN 4
            WHEN '91-180 days' THEN 5
            ELSE 6
        END
    """
).fetchdf()

display(inventory_summary)

,inventory_age_bucket,inventory_units,sold_units,unsold_units,average_days_to_sell,inventory_cost_value,inventory_retail_value
0,Sold,180862,180862.0,0.0,30.07,5194002.16,10796335.70
1,0-30 days,5061,0.0,5061.0,NaN,143051.21,298532.13
2,31-60 days,5893,0.0,5893.0,NaN,172185.77,357851.23
3,61-90 days,5788,0.0,5788.0,NaN,165430.61,344276.47
4,91-180 days,17720,0.0,17720.0,NaN,510283.29,1064714.32
5,181+ days,272822,0.0,272822.0,NaN,7842124.46,16297485.76


## 9. Build business marts

Execute the three Stage 8 SQL modules:

1. acquisition and funnel marts;
2. commercial and customer marts;
3. operations and inventory marts.

Marts contain question-ready aggregations. They do not replace the detailed core fact tables.

In [48]:
mart_sql_files = [
    "04_mart_acquisition_funnel.sql",
    "05_mart_commercial_customer.sql",
    "06_mart_operations_inventory.sql",
]

for file_name in mart_sql_files:
    execute_sql_file(file_name)

Executed successfully: 04_mart_acquisition_funnel.sql
Executed successfully: 05_mart_commercial_customer.sql
Executed successfully: 06_mart_operations_inventory.sql


In [49]:
#List all mart tables
mart_objects = connection.execute(
    """
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = 'mart'
    ORDER BY table_name
    """
).fetchdf()

display(mart_objects)

,table_schema,table_name,table_type
0,mart,acquisition_source_value,BASE TABLE
1,mart,cart_abandonment_segments,BASE TABLE
2,mart,channel_quality,BASE TABLE
3,mart,cohort_retention,BASE TABLE
4,mart,customer_360,BASE TABLE
5,mart,delivery_performance_dc,BASE TABLE
6,mart,funnel_monthly_channel,BASE TABLE
7,mart,funnel_stage_channel,BASE TABLE
8,mart,geography_performance_country,BASE TABLE
9,mart,inventory_performance,BASE TABLE


In [50]:
#Define the mart tables and grains
mart_grains = {
    "mart.funnel_stage_channel": [
        "traffic_source",
        "stage_order",
    ],
    "mart.funnel_monthly_channel": [
        "month_start",
        "traffic_source",
    ],
    "mart.cart_abandonment_segments": [
        "segment_type",
        "segment_value",
    ],
    "mart.channel_quality": [
        "traffic_source",
    ],
    "mart.acquisition_source_value": [
        "acquisition_source",
    ],
    "mart.sales_monthly": [
        "month_start",
    ],
    "mart.product_performance_category": [
        "department",
        "category",
    ],
    "mart.product_performance_brand": [
        "brand",
    ],
    "mart.geography_performance_country": [
        "country",
    ],
    "mart.customer_360": [
        "customer_id",
    ],
    "mart.cohort_retention": [
        "cohort_month",
        "months_since_first_order",
    ],
    "mart.operations_monthly": [
        "month_start",
    ],
    "mart.delivery_performance_dc": [
        "distribution_center_id",
    ],
    "mart.return_risk_segments": [
        "segment_type",
        "segment_value",
    ],
    "mart.inventory_performance": [
        "department",
        "category",
        "distribution_center_name",
    ],
}

In [51]:
#Check every mart’s grain
mart_grain_records = []

for table_name, key_columns in mart_grains.items():
    key_sql = ", ".join(key_columns)

    total_rows = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM {table_name}
        """
    ).fetchone()[0]

    duplicate_groups = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM (
            SELECT
                {key_sql},
                COUNT(*) AS rows_at_grain
            FROM {table_name}
            GROUP BY {key_sql}
            HAVING COUNT(*) > 1
        )
        """
    ).fetchone()[0]

    mart_grain_records.append(
        {
            "mart": table_name,
            "grain_columns": key_sql,
            "rows": total_rows,
            "duplicate_grain_groups": duplicate_groups,
            "test_result": (
                "PASS"
                if total_rows > 0
                and duplicate_groups == 0
                else "FAIL"
            ),
        }
    )

mart_grain_tests = pd.DataFrame(
    mart_grain_records
)

display(mart_grain_tests)

,mart,grain_columns,rows,duplicate_grain_groups,test_result
0,mart.funnel_stage_channel,"traffic_source, stage_order",20,0,PASS
1,mart.funnel_monthly_channel,"month_start, traffic_source",325,0,PASS
2,mart.cart_abandonment_segments,"segment_type, segment_value",22,0,PASS
3,mart.channel_quality,traffic_source,5,0,PASS
4,mart.acquisition_source_value,acquisition_source,5,0,PASS
5,mart.sales_monthly,month_start,65,0,PASS
6,mart.product_performance_category,"department, category",36,0,PASS
7,mart.product_performance_brand,brand,2755,0,PASS
8,mart.geography_performance_country,country,15,0,PASS
9,mart.customer_360,customer_id,80095,0,PASS


In [52]:
#Stop if a grain check fails
failed_mart_grain_tests = mart_grain_tests[
    mart_grain_tests["test_result"] != "PASS"
]

if failed_mart_grain_tests.empty:
    print(
        "PASS: all marts contain rows "
        "and are unique at their documented grains."
    )
else:
    display(failed_mart_grain_tests)

    raise AssertionError(
        "One or more mart grain tests failed."
    )

PASS: all marts contain rows and are unique at their documented grains.


In [53]:
#Preview every mart
for table_name in mart_grains:
    print(f"\n{'=' * 75}")
    print(table_name)
    print(f"{'=' * 75}")

    display(
        preview_table(
            table_name,
            row_limit=15,
        )
    )


mart.funnel_stage_channel


,traffic_source,stage_order,stage,stage_sessions,conversion_from_session_rate,conversion_from_previous_stage_rate
0,Facebook,1,Sessions,68518.0,1.000000,NaN
1,Facebook,2,Product Viewed,68518.0,1.000000,1.000000
2,Facebook,3,Cart Reached,43551.0,0.635614,0.635614
3,Facebook,4,Purchase,18343.0,0.267711,0.421184
4,YouTube,1,Sessions,68524.0,1.000000,NaN
5,YouTube,2,Product Viewed,68524.0,1.000000,1.000000
6,YouTube,3,Cart Reached,43385.0,0.633136,0.633136
7,YouTube,4,Purchase,18172.0,0.265192,0.418854
8,Organic,1,Sessions,33753.0,1.000000,NaN
9,Organic,2,Product Viewed,33753.0,1.000000,1.000000



mart.funnel_monthly_channel


,month_start,traffic_source,sessions,users,product_view_sessions,cart_sessions,purchase_sessions,abandoned_cart_sessions,session_conversion_rate,cart_abandonment_rate,avg_events_per_session,avg_session_duration_seconds,avg_event_identity_coverage_rate
0,2023-08-01,Adwords,4056,1495,4056.0,2895.0,1728.0,1167.0,0.426036,0.403109,4.345414,37614.805473,0.426036
1,2024-04-01,YouTube,2045,1166,2045.0,1628.0,1255.0,373.0,0.613692,0.229115,5.253790,57898.428851,0.613692
2,2024-04-01,Facebook,1985,1169,1985.0,1595.0,1237.0,358.0,0.623174,0.224451,5.331486,56951.247355,0.623174
3,2021-06-01,Email,4163,615,4163.0,2453.0,732.0,1721.0,0.175835,0.701590,3.085275,16645.601489,0.175835
4,2022-11-01,Adwords,3312,977,3312.0,2173.0,1106.0,1067.0,0.333937,0.491026,3.844807,30160.897645,0.333937
5,2023-06-01,Email,5547,1685,5547.0,3827.0,2055.0,1772.0,0.370471,0.463026,4.097350,34510.096989,0.370471
6,2022-12-01,Facebook,1129,337,1129.0,756.0,347.0,409.0,0.307352,0.541005,3.729849,24980.286094,0.307352
7,2022-12-01,Adwords,3522,976,3522.0,2311.0,1106.0,1205.0,0.314026,0.521419,3.753833,28328.556218,0.314026
8,2024-04-01,Adwords,6087,3190,6087.0,5001.0,3843.0,1158.0,0.631345,0.231554,5.407754,57695.262855,0.631345
9,2022-09-01,Adwords,3251,830,3251.0,2071.0,957.0,1114.0,0.294371,0.537904,3.681636,27995.556752,0.294371



mart.cart_abandonment_segments


...

,distribution_center_id,distribution_center_name,items,valid_timeline_items,avg_ship_lead_days,median_ship_lead_days,p90_ship_lead_days,avg_delivery_lead_days,median_delivery_lead_days,p90_delivery_lead_days,avg_end_to_end_lead_days,median_end_to_end_lead_days,p90_end_to_end_lead_days,observed_return_rate
0,10,Savannah GA,11929,9561.0,1.454967,1.410764,2.716042,2.489353,2.519444,4.505278,3.484969,3.427083,5.984028,0.287304
1,3,Houston TX,22514,18056.0,1.445936,1.391667,2.690764,2.459172,2.431944,4.504167,3.449433,3.352083,5.962083,0.293433
2,2,Chicago IL,24213,19415.0,1.466509,1.412500,2.722917,2.483195,2.460069,4.507708,3.473840,3.401042,5.970347,0.275412
3,9,Charleston SC,16550,13341.0,1.457754,1.403125,2.718819,2.458527,2.467361,4.487778,3.440039,3.353472,5.913056,0.290698
4,7,Philadelphia PA,16767,13500.0,1.443386,1.406250,2.688889,2.476382,2.463889,4.486111,3.449299,3.320139,5.954028,0.280666
5,5,New Orleans LA,12921,10474.0,1.470146,1.412153,2.733333,2.505541,2.509722,4.546806,3.535086,3.485417,6.007431,0.281381
6,1,Memphis TN,24065,19341.0,1.467875,1.427083,2.710417,2.508327,2.499306,4.530556,3.507688,3.439583,5.966667,0.284443
7,6,Port Authority of New York/New Jersey NY/NJ,16484,13190.0,1.463747,1.420486,2.725000,2.501667,2.479514,4.515625,3.470553,3.402083,5.965278,0.285152
8,4,Los Angeles CA,17094,13768.0,1.459257,1.409028,2.725000,2.495642,2.520486,4.494583,3.493107,3.439583,5.934028,0.280233
9,8,Mobile AL,18325,14776.0,1.465897,1.423611,2.732361,2.518432,2.510069,4.521597,3.497000,3.393056,6.029444,0.285179



mart.return_risk_segments


,segment_type,segment_value,items,eligible_items,returned_items,returned_value,observed_return_rate
0,Category,Underwear,7438,2598.0,728.0,19743.499998,0.280216
1,Category,Jumpsuits & Rompers,931,333.0,116.0,4915.100019,0.348348
2,Category,Pants,7237,2506.0,722.0,41672.660147,0.288109
3,Category,Socks & Hosiery,3832,1305.0,341.0,5854.109992,0.261303
4,Category,Shorts,11229,3923.0,1069.0,48854.220171,0.272496
5,Category,Blazers & Jackets,3128,1081.0,287.0,30757.050097,0.265495
6,Category,Skirts,2053,741.0,227.0,12696.090009,0.306343
7,Category,Fashion Hoodies & Sweatshirts,11855,4207.0,1172.0,64046.810066,0.278583
8,Category,Accessories,9946,3419.0,988.0,41100.949942,0.288973
9,Category,Plus,4300,1538.0,419.0,16647.160021,0.272432



mart.inventory_performance


,department,category,distribution_center_name,inventory_units,sold_units,unsold_units,sell_through_rate,median_days_to_sell,p90_days_to_sell,aged_181_plus_units,unsold_inventory_cost,unsold_inventory_retail_value
0,Men,Accessories,Houston TX,1909,708.0,1201.0,0.370875,30.0,54.0,1092.0,18865.689462,47218.580262
1,Women,Accessories,Charleston SC,744,272.0,472.0,0.365591,32.0,56.0,410.0,4602.929799,11296.589946
2,Women,Accessories,Houston TX,1445,536.0,909.0,0.370934,28.0,55.0,806.0,20792.017185,51659.839882
3,Women,Accessories,Philadelphia PA,670,254.0,416.0,0.379104,29.0,54.7,368.0,6243.312762,15524.620102
4,Men,Accessories,Philadelphia PA,1517,567.0,950.0,0.373764,31.0,55.0,843.0,8099.395926,20288.259994
5,Men,Accessories,Charleston SC,386,144.0,242.0,0.373057,30.0,56.0,217.0,3640.447655,9304.089925
6,Men,Active,Charleston SC,1382,508.0,874.0,0.367583,30.0,53.0,775.0,13553.692478,32279.019941
7,Men,Active,Houston TX,1881,702.0,1179.0,0.373206,32.0,54.0,1038.0,27553.549711,65137.699556
8,Women,Active,Los Angeles CA,1015,377.0,638.0,0.371429,28.0,55.0,581.0,14511.662796,34839.210134
9,Men,Active,Philadelphia PA,1047,385.0,662.0,0.367717,31.0,55.0,583.0,13890.668268,32635.479890


#Reconcile marts against facts

In [54]:
#Core reconciliation
mart_reconciliation = connection.execute(
    """
    WITH checks AS (

        SELECT
            'Channel-quality session count'
                AS check_name,

            (
                SELECT COUNT(*)::DOUBLE
                FROM core.fact_session
            ) AS expected_value,

            (
                SELECT SUM(sessions)::DOUBLE
                FROM mart.channel_quality
            ) AS actual_value

        UNION ALL

        SELECT
            'Channel-quality purchases',

            (
                SELECT SUM(purchase_flag)::DOUBLE
                FROM core.fact_session
            ),

            (
                SELECT SUM(purchase_sessions)::DOUBLE
                FROM mart.channel_quality
            )

        UNION ALL

        SELECT
            'Monthly-sales item count',

            (
                SELECT COUNT(*)::DOUBLE
                FROM core.fact_order_item
            ),

            (
                SELECT SUM(items)::DOUBLE
                FROM mart.sales_monthly
            )

        UNION ALL

        SELECT
            'Monthly gross sales',

            (
                SELECT ROUND(
                    SUM(gross_sales_value),
                    2
                )
                FROM core.fact_order_item
            ),

            (
                SELECT ROUND(
                    SUM(gross_sales_value),
                    2
                )
                FROM mart.sales_monthly
            )

        UNION ALL

        SELECT
            'Monthly net sales',

            (
                SELECT ROUND(
                    SUM(net_sales_value),
                    2
                )
                FROM core.fact_order_item
            ),

            (
                SELECT ROUND(
                    SUM(net_sales_value),
                    2
                )
                FROM mart.sales_monthly
            )

        UNION ALL

        SELECT
            'Monthly net profit',

            (
                SELECT ROUND(
                    SUM(net_profit_value),
                    2
                )
                FROM core.fact_order_item
            ),

            (
                SELECT ROUND(
                    SUM(net_profit_value),
                    2
                )
                FROM mart.sales_monthly
            )

        UNION ALL

        SELECT
            'Operations item count',

            (
                SELECT COUNT(*)::DOUBLE
                FROM core.fact_order_item
            ),

            (
                SELECT SUM(items)::DOUBLE
                FROM mart.operations_monthly
            )

        UNION ALL

        SELECT
            'Inventory-unit count',

            (
                SELECT COUNT(*)::DOUBLE
                FROM core.fact_inventory
            ),

            (
                SELECT SUM(inventory_units)::DOUBLE
                FROM mart.inventory_performance
            )

        UNION ALL

        SELECT
            'Sold inventory units',

            (
                SELECT SUM(sold_flag)::DOUBLE
                FROM core.fact_inventory
            ),

            (
                SELECT SUM(sold_units)::DOUBLE
                FROM mart.inventory_performance
            )
    )

    SELECT
        check_name,
        expected_value,
        actual_value,
        actual_value - expected_value
            AS difference,

        CASE
            WHEN ABS(
                COALESCE(actual_value, 0)
                - COALESCE(expected_value, 0)
            ) < 0.01
            THEN 'PASS'
            ELSE 'FAIL'
        END AS test_result

    FROM checks

    ORDER BY check_name
    """
).fetchdf()

display(mart_reconciliation)

,check_name,expected_value,actual_value,difference,test_result
0,Channel-quality purchases,180862.00,180862.00,0.0,PASS
1,Channel-quality session count,680862.00,680862.00,0.0,PASS
2,Inventory-unit count,488146.00,488146.00,0.0,PASS
3,Monthly gross sales,10796335.70,10796335.70,0.0,PASS
4,Monthly net profit,4201367.01,4201367.01,0.0,PASS
5,Monthly net sales,8096531.48,8096531.48,0.0,PASS
6,Monthly-sales item count,180862.00,180862.00,0.0,PASS
7,Operations item count,180862.00,180862.00,0.0,PASS
8,Sold inventory units,180862.00,180862.00,0.0,PASS


In [55]:
#Stop if reconciliation fails
failed_mart_reconciliation = mart_reconciliation[
    mart_reconciliation["test_result"] != "PASS"
]

if failed_mart_reconciliation.empty:
    print(
        "PASS: mart totals reconcile "
        "to their source fact tables."
    )
else:
    display(failed_mart_reconciliation)

    raise AssertionError(
        "One or more mart totals do not reconcile."
    )

PASS: mart totals reconcile to their source fact tables.


Focused business previews

In [56]:
#Monthly sales
monthly_sales_preview = connection.execute(
    """
    SELECT *
    FROM mart.sales_monthly
    ORDER BY month_start DESC
    LIMIT 12
    """
).fetchdf()

display(monthly_sales_preview)

,month_start,year,month_number,is_complete_month,orders,customers,items,gross_sales_value,cancelled_value,returned_value,net_sales_value,net_profit_value,net_margin_rate,average_order_value,items_per_order,item_cancellation_rate,item_return_rate
0,2024-05-01,2024,5,0,6556,4973,9693,579392.630463,82921.620029,59139.340080,437331.670353,226488.913030,0.517888,66.707088,1.478493,0.139895,0.102445
1,2024-04-01,2024,4,1,8703,7656,12313,748773.540782,120729.520161,77886.140104,550157.880517,285251.263310,0.518490,63.214740,1.414799,0.153090,0.106716
2,2024-03-01,2024,3,1,7122,6568,10040,594568.650664,93183.620128,60050.820075,441334.210461,229097.540089,0.519102,61.967735,1.409716,0.152888,0.098008
3,2024-02-01,2024,2,1,5813,5500,8226,496183.940308,70339.590047,43871.749996,381972.600265,198563.687004,0.519838,65.710064,1.415104,0.146244,0.098711
4,2024-01-01,2024,1,1,5539,5307,7796,468036.380438,73119.920108,46178.720032,348737.740298,180965.920185,0.518917,62.960415,1.407474,0.156491,0.101077
5,2023-12-01,2023,12,1,5221,5013,7340,432875.970530,64659.480012,41914.830063,326301.660455,168954.891875,0.517787,62.497924,1.405861,0.146049,0.096049
6,2023-11-01,2023,11,1,4525,4363,6510,390124.900493,55312.490162,36344.730053,298467.680278,155330.051792,0.520425,65.959708,1.438674,0.143932,0.094316
7,2023-10-01,2023,10,1,4447,4320,6246,374666.580396,57903.010044,36781.950071,279981.620282,145193.221509,0.518581,62.959663,1.404542,0.147454,0.099264
8,2023-09-01,2023,9,1,4091,3986,5762,348572.650347,46579.430061,40854.699979,261138.520307,135291.184018,0.518082,63.832442,1.408458,0.144047,0.115411
9,2023-08-01,2023,8,1,3947,3837,5675,331357.400305,53132.640093,32094.990025,246129.770188,126847.267465,0.515367,62.358695,1.437801,0.153304,0.098150


In [57]:
#Channel quality
channel_quality_preview = connection.execute(
    """
    SELECT
        traffic_source,
        sessions,
        users,
        purchase_sessions,
        session_conversion_rate,
        cart_abandonment_rate,
        avg_events_per_session,
        avg_event_identity_coverage_rate
    FROM mart.channel_quality
    ORDER BY session_conversion_rate DESC
    """
).fetchdf()

display(channel_quality_preview)

,traffic_source,sessions,users,purchase_sessions,session_conversion_rate,cart_abandonment_rate,avg_events_per_session,avg_event_identity_coverage_rate
0,Facebook,68518,16366,18343.0,0.267711,0.578816,3.563166,0.267711
1,Email,306377,52125,81364.0,0.265568,0.579953,3.556615,0.265568
2,Adwords,203690,39552,54072.0,0.265462,0.579697,3.552796,0.265462
3,YouTube,68524,16189,18172.0,0.265192,0.581146,3.554638,0.265192
4,Organic,33753,8380,8911.0,0.264006,0.582153,3.543626,0.264006


In [58]:
#Customer value
customer_value_preview = connection.execute(
    """
    SELECT
        customer_id,
        country,
        acquisition_source,
        recency_days,
        order_count,
        item_count,
        net_sales_value,
        net_profit_value,
        average_order_value,
        repeat_customer_flag
    FROM mart.customer_360
    ORDER BY net_sales_value DESC
    LIMIT 20
    """
).fetchdf()

display(customer_value_preview)

,customer_id,country,acquisition_source,recency_days,order_count,item_count,net_sales_value,net_profit_value,average_order_value,repeat_customer_flag
0,71666,China,Search,487,4,7,1747.290016,990.030079,436.822504,1
1,60104,Belgium,Search,574,4,4,1399.450001,742.257848,349.862500,1
2,40179,China,Search,558,2,7,1347.769993,767.663844,673.884996,1
3,77435,China,Email,25,2,6,1315.950001,724.854149,657.975000,1
4,90092,Spain,Search,129,4,8,1303.599993,701.130586,325.899998,1
5,83700,South Korea,Organic,732,2,8,1295.980000,732.447998,647.990000,1
6,9805,Brasil,Search,788,3,5,1294.250000,587.834998,431.416667,1
7,78362,United States,Search,196,4,12,1289.960003,699.097351,322.490001,1
8,36442,Brasil,Search,203,2,3,1289.490002,720.294270,644.745001,1
9,61698,Spain,Search,112,2,7,1278.999999,735.281218,639.500000,1


In [59]:
#Cohort retention
cohort_retention_preview = connection.execute(
    """
    SELECT *
    FROM mart.cohort_retention
    ORDER BY
        cohort_month,
        months_since_first_order
    LIMIT 50
    """
).fetchdf()

display(cohort_retention_preview)

,cohort_month,months_since_first_order,cohort_size,active_customers,retention_rate
0,2019-01-01,0,10,10,1.000000
1,2019-01-01,1,10,2,0.200000
2,2019-01-01,6,10,1,0.100000
3,2019-01-01,8,10,1,0.100000
4,2019-01-01,10,10,1,0.100000
5,2019-01-01,13,10,1,0.100000
6,2019-01-01,21,10,1,0.100000
7,2019-01-01,32,10,1,0.100000
8,2019-01-01,38,10,1,0.100000
9,2019-01-01,47,10,1,0.100000


In [60]:
#Operations
operations_preview = connection.execute(
    """
    SELECT *
    FROM mart.operations_monthly
    ORDER BY month_start DESC
    LIMIT 12
    """
).fetchdf()

display(operations_preview)

,month_start,items,valid_timeline_items,avg_ship_lead_days,median_ship_lead_days,p90_ship_lead_days,avg_delivery_lead_days,median_delivery_lead_days,p90_delivery_lead_days,avg_end_to_end_lead_days,median_end_to_end_lead_days,p90_end_to_end_lead_days,observed_return_rate,cancellation_rate
0,2024-05-01,9693,7514.0,1.429263,1.374653,2.667431,2.497351,2.478472,4.481250,3.391441,3.304167,5.851736,0.295272,0.139895
1,2024-04-01,12313,9984.0,1.488685,1.449306,2.743194,2.502766,2.490278,4.515972,3.567330,3.507986,6.031181,0.304802,0.153090
2,2024-03-01,10040,8246.0,1.497065,1.495139,2.722222,2.464165,2.444444,4.494861,3.557819,3.504861,6.052083,0.287132,0.152888
3,2024-02-01,8226,6652.0,1.433858,1.388542,2.699306,2.494685,2.492361,4.506944,3.449744,3.344792,5.931250,0.282730,0.146244
4,2024-01-01,7796,6352.0,1.452529,1.365972,2.715000,2.503722,2.513194,4.484722,3.490505,3.411458,5.930556,0.295907,0.156491
5,2023-12-01,7340,5932.0,1.437719,1.381597,2.696181,2.506362,2.513889,4.474306,3.490959,3.422222,5.970139,0.274212,0.146049
6,2023-11-01,6510,5135.0,1.465388,1.414583,2.720139,2.441129,2.395833,4.547014,3.339708,3.225694,5.871528,0.270246,0.143932
7,2023-10-01,6246,5042.0,1.477652,1.414583,2.747083,2.444774,2.356944,4.504861,3.457634,3.380208,5.902431,0.285583,0.147454
8,2023-09-01,5762,4638.0,1.494974,1.481597,2.722222,2.547913,2.620833,4.441667,3.554867,3.471528,5.947361,0.313679,0.144047
9,2023-08-01,5675,4521.0,1.437989,1.400347,2.713403,2.477857,2.456944,4.534028,3.466099,3.422569,5.932986,0.275606,0.153304


In [61]:
#Return-risk segments
return_risk_preview = connection.execute(
    """
    SELECT *
    FROM mart.return_risk_segments
    ORDER BY
        observed_return_rate DESC,
        eligible_items DESC
    LIMIT 20
    """
).fetchdf()

display(return_risk_preview)

,segment_type,segment_value,items,eligible_items,returned_items,returned_value,observed_return_rate
0,Category,Jumpsuits & Rompers,931,333.0,116.0,4915.100019,0.348348
1,Category,Skirts,2053,741.0,227.0,12696.090009,0.306343
2,Country,Spain,7416,2627.0,800.0,47959.650008,0.304530
3,Category,Pants & Capris,3415,1215.0,360.0,20248.720120,0.296296
4,Category,Socks,6229,2152.0,635.0,12025.640003,0.295074
5,Category,Outerwear & Coats,8916,3121.0,920.0,132444.719954,0.294777
6,Age Band,55-64,30991,10851.0,3187.0,185519.760284,0.293706
7,Category,Sweaters,10959,3878.0,1138.0,84109.679993,0.293450
8,Country,South Korea,9573,3313.0,969.0,63818.520064,0.292484
9,Age Band,45-54,30580,10657.0,3106.0,184232.440302,0.291452


In [62]:
#Inventory performance
inventory_performance_preview = connection.execute(
    """
    SELECT *
    FROM mart.inventory_performance
    ORDER BY unsold_inventory_cost DESC
    LIMIT 20
    """
).fetchdf()

display(inventory_performance_preview)

,department,category,distribution_center_name,inventory_units,sold_units,unsold_units,sell_through_rate,median_days_to_sell,p90_days_to_sell,aged_181_plus_units,unsold_inventory_cost,unsold_inventory_retail_value
0,Men,Outerwear & Coats,Houston TX,3479,1285.0,2194.0,0.369359,30.0,55.0,1941.0,189522.858455,432467.968418
1,Men,Jeans,Savannah GA,2914,1078.0,1836.0,0.369938,29.0,54.0,1629.0,142119.161532,267436.139641
2,Men,Jeans,Philadelphia PA,4040,1482.0,2558.0,0.366832,30.0,53.0,2289.0,137680.330264,258290.490812
3,Women,Jeans,Mobile AL,3130,1151.0,1979.0,0.367732,30.0,54.0,1774.0,116017.378389,215118.920425
4,Men,Jeans,Mobile AL,2698,1002.0,1696.0,0.371386,30.0,54.0,1504.0,106601.064671,200268.810404
5,Women,Dresses,Houston TX,2981,1098.0,1883.0,0.368333,30.0,54.0,1659.0,105772.947844,236431.640156
6,Women,Outerwear & Coats,Houston TX,1671,616.0,1055.0,0.368642,28.0,54.0,923.0,90454.799399,200409.929913
7,Women,Jeans,Philadelphia PA,1723,629.0,1094.0,0.365061,28.0,53.2,976.0,89461.856407,164544.739958
8,Women,Intimates,Chicago IL,7598,2807.0,4791.0,0.369439,30.0,54.0,4256.0,84105.635255,157697.119877
9,Men,Outerwear & Coats,Memphis TN,1979,729.0,1250.0,0.368368,29.0,54.0,1099.0,82799.374026,188622.160608


# Stage 9 — QA, warnings, and metric reconciliation

This section:

1. executes `07_quality_and_snapshots.sql`;
2. reviews critical failures;
3. reviews controlled warnings;
4. verifies metric reconciliation;
5. previews the current warehouse snapshot.

In [63]:
#connect to DuckDB
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

project_root = Path(
    r"C:\Users\phanh\data_analysis\p_projects"
    r"\the_look_ecommerce\practice_analysis"
)

database_path = (
    project_root
    / "artifacts"
    / "practice_analytics.duckdb"
)

connection = duckdb.connect(str(database_path))

print(f"Connected to: {database_path}")

Connected to: C:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\practice_analysis\artifacts\practice_analytics.duckdb


In [64]:
#execute Stage 9
qa_sql_path = (
    project_root
    / "sql"
    / "duckdb"
    / "07_quality_and_snapshots.sql"
)

qa_sql = qa_sql_path.read_text(
    encoding="utf-8"
)

connection.execute(qa_sql)

print("07_quality_and_snapshots.sql executed successfully.")

07_quality_and_snapshots.sql executed successfully.


In [65]:
#show all QA results
qa_results = connection.execute(
    """
    SELECT
        check_name,
        layer,
        severity,
        status,
        failure_count,
        denominator,
        ROUND(failure_rate * 100, 4) AS failure_percentage,
        handling_rule
    FROM qa.test_results
    ORDER BY
        CASE status
            WHEN 'FAIL' THEN 1
            WHEN 'WARN' THEN 2
            WHEN 'INFO' THEN 3
            WHEN 'PASS' THEN 4
        END,
        check_name
    """
).df()

display(qa_results)

,check_name,layer,severity,status,failure_count,denominator,failure_percentage,handling_rule
0,invalid_order_item_timeline,core,HIGH,WARN,35440,180862,19.5951,Exclude invalid timelines from lead-time metri...
1,events_without_user_id,staging,INFO,INFO,1124527,2420661,46.4554,Anonymous events remain valid for traffic anal...
2,rows_in_incomplete_maximum_month,core,INFO,INFO,13,1994,0.6520,Exclude the incomplete maximum month from comp...
3,blank_event_session_ids,staging,WARNING,PASS,0,2420661,0.0000,Events without session_id are excluded from se...
4,cost_above_sale_price,core,WARNING,PASS,0,180862,0.0000,Review potentially loss-making items; do not s...
5,dim_customer_key_unique,core,CRITICAL,PASS,0,100000,0.0000,Every customer must have one unique user_id
6,dim_date_key_unique,core,CRITICAL,PASS,0,1994,0.0000,Every calendar date must have one unique date_key
7,dim_distribution_center_key_unique,core,CRITICAL,PASS,0,10,0.0000,Every distribution center must have one unique ID
8,dim_product_key_unique,core,CRITICAL,PASS,0,29120,0.0000,Every product must have one unique product_id
9,fact_inventory_key_unique,core,CRITICAL,PASS,0,488146,0.0000,Every inventory item must appear once in fact_...


In [66]:
#blocking QA gate
blocking_failures = connection.execute(
    """
    SELECT
        check_name,
        layer,
        severity,
        failure_count,
        denominator,
        handling_rule
    FROM qa.test_results
    WHERE status = 'FAIL'
      AND blocking = 1
    ORDER BY severity, check_name
    """
).df()

if blocking_failures.empty:
    print("PASS: No blocking QA failures.")
else:
    display(blocking_failures)

    raise AssertionError(
        f"{len(blocking_failures)} blocking QA check(s) failed. "
        "Do not publish the Power BI outputs yet."
    )

PASS: No blocking QA failures.


In [67]:
#review warnings
warnings = connection.execute(
    """
    SELECT
        check_name,
        failure_count,
        denominator,
        ROUND(failure_rate * 100, 4) AS failure_percentage,
        handling_rule
    FROM qa.test_results
    WHERE status = 'WARN'
    ORDER BY failure_rate DESC, check_name
    """
).df()

if warnings.empty:
    print("No controlled warnings were found.")
else:
    print(
        "Warnings do not automatically stop publication, "
        "but each one must be documented."
    )

    display(warnings)

Warnings do not automatically stop publication, but each one must be documented.


,check_name,failure_count,denominator,failure_percentage,handling_rule
0,invalid_order_item_timeline,35440,180862,19.5951,Exclude invalid timelines from lead-time metri...


In [68]:
#review informational observations
information = connection.execute(
    """
    SELECT
        check_name,
        failure_count AS observed_rows,
        denominator,
        ROUND(failure_rate * 100, 4) AS observed_percentage,
        handling_rule
    FROM qa.test_results
    WHERE status = 'INFO'
    ORDER BY check_name
    """
).df()

display(information)

,check_name,observed_rows,denominator,observed_percentage,handling_rule
0,events_without_user_id,1124527,2420661,46.4554,Anonymous events remain valid for traffic anal...
1,rows_in_incomplete_maximum_month,13,1994,0.6520,Exclude the incomplete maximum month from comp...


In [69]:
#metric reconciliation
reconciliation = connection.execute(
    """
    SELECT
        metric_name,
        source_layer,
        modeled_layer,
        source_value,
        modeled_value,
        variance,
        tolerance,
        status
    FROM qa.metric_reconciliation
    ORDER BY status, metric_name
    """
).df()

display(reconciliation)

,metric_name,source_layer,modeled_layer,source_value,modeled_value,variance,tolerance,status
0,events_into_sessions,stg.events,core.fact_session,2.420661e+06,2.420661e+06,0.000000e+00,0.00,PASS
1,gross_sales_fact_to_monthly_mart,core.fact_order_item,mart.sales_monthly,1.079634e+07,1.079634e+07,0.000000e+00,0.01,PASS
2,gross_sales_value,stg.order_items,core.fact_order_item,1.079634e+07,1.079634e+07,0.000000e+00,0.01,PASS
3,inventory_rows,stg.inventory_events,core.fact_inventory,4.881460e+05,4.881460e+05,0.000000e+00,0.00,PASS
4,net_profit_fact_to_monthly_mart,core.fact_order_item,mart.sales_monthly,4.201367e+06,4.201367e+06,9.313226e-09,0.01,PASS
5,net_sales_fact_to_monthly_mart,core.fact_order_item,mart.sales_monthly,8.096531e+06,8.096531e+06,0.000000e+00,0.01,PASS
6,order_items_rows,stg.order_items,core.fact_order_item,1.808620e+05,1.808620e+05,0.000000e+00,0.00,PASS
7,orders_rows,stg.orders,core.fact_order,1.248140e+05,1.248140e+05,0.000000e+00,0.00,PASS
8,sessions_from_events,stg.events,core.fact_session,6.808620e+05,6.808620e+05,0.000000e+00,0.00,PASS


In [70]:
#reconciliation gate
failed_reconciliations = connection.execute(
    """
    SELECT *
    FROM qa.metric_reconciliation
    WHERE status = 'FAIL'
    ORDER BY metric_name
    """
).df()

if failed_reconciliations.empty:
    print("PASS: Every reconciled metric is within tolerance.")
else:
    display(failed_reconciliations)

    raise AssertionError(
        f"{len(failed_reconciliations)} metric reconciliation(s) failed."
    )

PASS: Every reconciled metric is within tolerance.


In [71]:
#current warehouse snapshot
metric_snapshot = connection.execute(
    """
    SELECT *
    FROM qa.metric_snapshot
    """
).df()

display(metric_snapshot)

,snapshot_at,customers,products,orders,order_items,sessions,inventory_units,gross_sales_value,net_sales_value,net_profit_value,returned_items,cancelled_items
0,2026-08-10 11:06:26.063171+07:00,100000,29120,124814,180862,680862,488146,1.079634e+07,8.096531e+06,4.201367e+06,18041.0,27072.0


In [72]:
#concise final status
qa_summary = connection.execute(
    """
    SELECT
        status,
        COUNT(*) AS checks
    FROM qa.test_results
    GROUP BY status
    ORDER BY status
    """
).df()

reconciliation_summary = connection.execute(
    """
    SELECT
        status,
        COUNT(*) AS metrics
    FROM qa.metric_reconciliation
    GROUP BY status
    ORDER BY status
    """
).df()

print("QA summary:")
display(qa_summary)

print("Reconciliation summary:")
display(reconciliation_summary)

QA summary:


,status,checks
0,INFO,2
1,PASS,19
2,WARN,1


Reconciliation summary:


,status,metrics
0,PASS,9


## Stage 9 — Full inspection of the three QA tables

Read each table end-to-end to understand the logic behind file 07:

1. `qa.test_results` — one row per check
2. `qa.metric_reconciliation` — one row per metric
3. `qa.metric_snapshot` — one row total (run scorecard)

In [73]:
# Cell 1 — (re)build the three QA tables
execute_sql_file("07_quality_and_snapshots.sql")

# Confirm all three objects now exist in the qa schema
connection.execute(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'qa'
    ORDER BY table_name
    """
).df()

Executed successfully: 07_quality_and_snapshots.sql


,table_name
0,metric_reconciliation
1,metric_snapshot
2,test_results


In [74]:
# Cell 2 — qa.test_results in full (schema -> all rows -> status counts)
from IPython.display import display

# (a) SCHEMA: what columns file 07 produces
print("--- qa.test_results columns ---")
display(describe_table("qa.test_results"))

# (b) EVERY row, ordered so failures surface first
print("--- qa.test_results (all rows) ---")
display(
    connection.execute(
        """
        SELECT
            check_name,
            layer,
            severity,
            status,
            blocking,
            failure_count,
            denominator,
            ROUND(failure_rate * 100, 4) AS failure_pct,
            handling_rule
        FROM qa.test_results
        ORDER BY
            CASE status WHEN 'FAIL' THEN 1 WHEN 'WARN' THEN 2
                        WHEN 'INFO' THEN 3 WHEN 'PASS' THEN 4 END,
            severity,
            check_name
        """
    ).df()
)

# (c) The STATUS logic in one glance
print("--- status counts ---")
display(
    connection.execute(
        """
        SELECT status, COUNT(*) AS checks
        FROM qa.test_results
        GROUP BY status
        ORDER BY status
        """
    ).df()
)

--- qa.test_results columns ---


,column_name,column_type,null,key,default,extra
0,check_name,VARCHAR,YES,None,None,None
1,layer,VARCHAR,YES,None,None,None
2,severity,VARCHAR,YES,None,None,None
3,failure_count,BIGINT,YES,None,None,None
4,denominator,BIGINT,YES,None,None,None
5,blocking,INTEGER,YES,None,None,None
6,handling_rule,VARCHAR,YES,None,None,None
7,status,VARCHAR,YES,None,None,None
8,failure_rate,DOUBLE,YES,None,None,None


--- qa.test_results (all rows) ---


,check_name,layer,severity,status,blocking,failure_count,denominator,failure_pct,handling_rule
0,invalid_order_item_timeline,core,HIGH,WARN,0,35440,180862,19.5951,Exclude invalid timelines from lead-time metri...
1,events_without_user_id,staging,INFO,INFO,0,1124527,2420661,46.4554,Anonymous events remain valid for traffic anal...
2,rows_in_incomplete_maximum_month,core,INFO,INFO,0,13,1994,0.6520,Exclude the incomplete maximum month from comp...
3,dim_customer_key_unique,core,CRITICAL,PASS,1,0,100000,0.0000,Every customer must have one unique user_id
4,dim_date_key_unique,core,CRITICAL,PASS,1,0,1994,0.0000,Every calendar date must have one unique date_key
5,dim_distribution_center_key_unique,core,CRITICAL,PASS,1,0,10,0.0000,Every distribution center must have one unique ID
6,dim_product_key_unique,core,CRITICAL,PASS,1,0,29120,0.0000,Every product must have one unique product_id
7,fact_inventory_key_unique,core,CRITICAL,PASS,1,0,488146,0.0000,Every inventory item must appear once in fact_...
8,fact_order_item_key_unique,core,CRITICAL,PASS,1,0,180862,0.0000,Every order item must appear once in fact_orde...
9,fact_order_key_unique,core,CRITICAL,PASS,1,0,124814,0.0000,Every order must appear once in fact_order


--- status counts ---


,status,checks
0,INFO,2
1,PASS,19
2,WARN,1


In [75]:
# Cell 3 — blocking gate for qa.test_results
blocking_failures = connection.execute(
    """
    SELECT check_name, layer, severity, failure_count, denominator, handling_rule
    FROM qa.test_results
    WHERE status = 'FAIL' AND blocking = 1
    ORDER BY severity, check_name
    """
).df()

if blocking_failures.empty:
    print("PASS: no blocking QA failures - safe to continue.")
else:
    display(blocking_failures)
    raise AssertionError(
        f"{len(blocking_failures)} blocking QA check(s) failed."
    )

PASS: no blocking QA failures - safe to continue.


In [76]:
# Cell 4 — qa.metric_reconciliation in full
# (a) SCHEMA
print("--- qa.metric_reconciliation columns ---")
display(describe_table("qa.metric_reconciliation"))

# (b) EVERY reconciliation row: source value vs modeled value
print("--- qa.metric_reconciliation (all rows) ---")
display(
    connection.execute(
        """
        SELECT
            metric_name,
            source_layer,
            modeled_layer,
            source_value,
            modeled_value,
            variance,          -- source_value - modeled_value
            tolerance,         -- 0.0 for counts, 0.01 for money
            status             -- PASS if ABS(variance) <= tolerance
        FROM qa.metric_reconciliation
        ORDER BY status, metric_name
        """
    ).df()
)

--- qa.metric_reconciliation columns ---


,column_name,column_type,null,key,default,extra
0,metric_name,VARCHAR,YES,None,None,None
1,source_layer,VARCHAR,YES,None,None,None
2,modeled_layer,VARCHAR,YES,None,None,None
3,source_value,DOUBLE,YES,None,None,None
4,modeled_value,DOUBLE,YES,None,None,None
5,variance,DOUBLE,YES,None,None,None
6,tolerance,"DECIMAL(3,2)",YES,None,None,None
7,status,VARCHAR,YES,None,None,None


--- qa.metric_reconciliation (all rows) ---


,metric_name,source_layer,modeled_layer,source_value,modeled_value,variance,tolerance,status
0,events_into_sessions,stg.events,core.fact_session,2.420661e+06,2.420661e+06,0.000000e+00,0.00,PASS
1,gross_sales_fact_to_monthly_mart,core.fact_order_item,mart.sales_monthly,1.079634e+07,1.079634e+07,0.000000e+00,0.01,PASS
2,gross_sales_value,stg.order_items,core.fact_order_item,1.079634e+07,1.079634e+07,0.000000e+00,0.01,PASS
3,inventory_rows,stg.inventory_events,core.fact_inventory,4.881460e+05,4.881460e+05,0.000000e+00,0.00,PASS
4,net_profit_fact_to_monthly_mart,core.fact_order_item,mart.sales_monthly,4.201367e+06,4.201367e+06,9.313226e-09,0.01,PASS
5,net_sales_fact_to_monthly_mart,core.fact_order_item,mart.sales_monthly,8.096531e+06,8.096531e+06,0.000000e+00,0.01,PASS
6,order_items_rows,stg.order_items,core.fact_order_item,1.808620e+05,1.808620e+05,0.000000e+00,0.00,PASS
7,orders_rows,stg.orders,core.fact_order,1.248140e+05,1.248140e+05,0.000000e+00,0.00,PASS
8,sessions_from_events,stg.events,core.fact_session,6.808620e+05,6.808620e+05,0.000000e+00,0.00,PASS


In [77]:
# Cell 5 — reconciliation gate
failed = connection.execute(
    """
    SELECT *
    FROM qa.metric_reconciliation
    WHERE status = 'FAIL'
    ORDER BY metric_name
    """
).df()

if failed.empty:
    print("PASS: every reconciled metric is within tolerance.")
else:
    display(failed)
    raise AssertionError(
        f"{len(failed)} metric reconciliation(s) failed."
    )

PASS: every reconciled metric is within tolerance.


In [78]:
# Cell 6 — qa.metric_snapshot in full (transposed so it reads top-to-bottom)
# (a) SCHEMA
print("--- qa.metric_snapshot columns ---")
display(describe_table("qa.metric_snapshot"))

# (b) Single-row run scorecard -> transpose for readability
snapshot = connection.execute("SELECT * FROM qa.metric_snapshot").df()
print("--- qa.metric_snapshot (this run) ---")
display(snapshot.T.rename(columns={0: "value"}))

--- qa.metric_snapshot columns ---


,column_name,column_type,null,key,default,extra
0,snapshot_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None
1,customers,BIGINT,YES,None,None,None
2,products,BIGINT,YES,None,None,None
3,orders,BIGINT,YES,None,None,None
4,order_items,BIGINT,YES,None,None,None
5,sessions,BIGINT,YES,None,None,None
6,inventory_units,BIGINT,YES,None,None,None
7,gross_sales_value,DOUBLE,YES,None,None,None
8,net_sales_value,DOUBLE,YES,None,None,None
9,net_profit_value,DOUBLE,YES,None,None,None


--- qa.metric_snapshot (this run) ---


,value
snapshot_at,2026-08-10 11:06:28.723123+07:00
customers,100000
products,29120
orders,124814
order_items,180862
sessions,680862
inventory_units,488146
gross_sales_value,10796335.701331
net_sales_value,8096531.478517
net_profit_value,4201367.012843


In [79]:
timeline_diagnostics = connection.execute(
    """
    SELECT
        COUNT(*) AS total_items,

        SUM(
            CASE
                WHEN shipped_at IS NOT NULL
                 AND shipped_at < order_item_created_at
                THEN 1 ELSE 0
            END
        ) AS shipped_before_created,

        SUM(
            CASE
                WHEN delivered_at IS NOT NULL
                 AND shipped_at IS NULL
                THEN 1 ELSE 0
            END
        ) AS delivered_without_shipping,

        SUM(
            CASE
                WHEN delivered_at IS NOT NULL
                 AND shipped_at IS NOT NULL
                 AND delivered_at < shipped_at
                THEN 1 ELSE 0
            END
        ) AS delivered_before_shipped,

        SUM(
            CASE
                WHEN returned_at IS NOT NULL
                 AND delivered_at IS NULL
                THEN 1 ELSE 0
            END
        ) AS returned_without_delivery,

        SUM(
            CASE
                WHEN returned_at IS NOT NULL
                 AND delivered_at IS NOT NULL
                 AND returned_at < delivered_at
                THEN 1 ELSE 0
            END
        ) AS returned_before_delivered,

        SUM(
            CASE
                WHEN valid_timeline_flag = 0
                THEN 1 ELSE 0
            END
        ) AS invalid_timeline_items

    FROM core.fact_order_item
    """
).df()

display(timeline_diagnostics)

,total_items,shipped_before_created,delivered_without_shipping,delivered_before_shipped,returned_without_delivery,returned_before_delivered,invalid_timeline_items
0,180862,35440.0,0.0,0.0,0.0,0.0,35440.0


In [80]:
#Check invalid timelines by status
timeline_diagnostics = connection.execute(
    """
    SELECT
        COUNT(*) AS total_items,

        SUM(
            CASE
                WHEN shipped_at IS NOT NULL
                 AND shipped_at < order_item_created_at
                THEN 1 ELSE 0
            END
        ) AS shipped_before_created,

        SUM(
            CASE
                WHEN delivered_at IS NOT NULL
                 AND shipped_at IS NULL
                THEN 1 ELSE 0
            END
        ) AS delivered_without_shipping,

        SUM(
            CASE
                WHEN delivered_at IS NOT NULL
                 AND shipped_at IS NOT NULL
                 AND delivered_at < shipped_at
                THEN 1 ELSE 0
            END
        ) AS delivered_before_shipped,

        SUM(
            CASE
                WHEN returned_at IS NOT NULL
                 AND delivered_at IS NULL
                THEN 1 ELSE 0
            END
        ) AS returned_without_delivery,

        SUM(
            CASE
                WHEN returned_at IS NOT NULL
                 AND delivered_at IS NOT NULL
                 AND returned_at < delivered_at
                THEN 1 ELSE 0
            END
        ) AS returned_before_delivered,

        SUM(
            CASE
                WHEN valid_timeline_flag = 0
                THEN 1 ELSE 0
            END
        ) AS invalid_timeline_items

    FROM core.fact_order_item
    """
).df()

display(timeline_diagnostics)

,total_items,shipped_before_created,delivered_without_shipping,delivered_before_shipped,returned_without_delivery,returned_before_delivered,invalid_timeline_items
0,180862,35440.0,0.0,0.0,0.0,0.0,35440.0


In [81]:
#Inspect examples
invalid_samples = connection.execute(
    """
    SELECT
        order_item_id,
        order_id,
        item_status,
        order_item_created_at,
        shipped_at,
        delivered_at,
        returned_at,
        ship_lead_days,
        delivery_lead_days,
        end_to_end_lead_days,
        valid_timeline_flag

    FROM core.fact_order_item

    WHERE valid_timeline_flag = 0

    LIMIT 20
    """
).df()

display(invalid_samples)

,order_item_id,order_id,item_status,order_item_created_at,shipped_at,delivered_at,returned_at,ship_lead_days,delivery_lead_days,end_to_end_lead_days,valid_timeline_flag
0,129627,89326,Complete,2023-10-28 01:49:38.000000,2023-10-27 08:52:00.000000,2023-10-30 11:06:00.000000,NaT,NaN,3.093056,2.386806,0
1,132606,91436,Complete,2024-04-28 16:57:50.000000,2024-04-27 07:29:00.000000,2024-05-02 01:13:00.000000,NaT,NaN,4.738889,3.344444,0
2,138256,95390,Complete,2024-03-14 13:05:47.000000,2024-03-13 17:01:00.000000,2024-03-16 09:57:00.000000,NaT,NaN,2.705556,1.869444,0
3,138874,95808,Complete,2023-11-08 10:33:37.000000,2023-11-06 15:47:00.000000,2023-11-07 16:14:00.000000,NaT,NaN,1.018750,NaN,0
4,139190,96013,Complete,2023-08-16 13:23:27.000000,2023-08-16 08:29:00.000000,2023-08-19 12:30:00.000000,NaT,NaN,3.167361,2.963194,0
5,140702,97071,Complete,2022-01-20 00:14:29.000000,2022-01-17 18:28:00.000000,2022-01-18 20:29:00.000000,NaT,NaN,1.084028,NaN,0
6,140924,97232,Complete,2022-01-06 02:45:27.000000,2022-01-04 08:29:00.000000,2022-01-09 06:12:00.000000,NaT,NaN,4.904861,3.143750,0
7,141928,97933,Complete,2022-05-21 22:56:38.000000,2022-05-19 20:15:00.000000,2022-05-24 17:32:00.000000,NaT,NaN,4.886806,2.775000,0
8,143208,98815,Complete,2019-06-29 11:24:51.000000,2019-06-27 23:20:00.000000,2019-07-02 06:18:00.000000,NaT,NaN,4.290278,2.787500,0
9,146480,101077,Complete,2024-02-17 16:24:15.000000,2024-02-17 10:01:00.000000,2024-02-22 06:51:00.000000,NaT,NaN,4.868056,4.602083,0


In [82]:
#Run the missing status analysis
invalid_by_status = connection.execute(
    """
    SELECT
        item_status,
        COUNT(*) AS total_items,

        SUM(
            CASE
                WHEN valid_timeline_flag = 0
                THEN 1 ELSE 0
            END
        ) AS invalid_items,

        SUM(
            CASE
                WHEN valid_timeline_flag = 0
                THEN 1 ELSE 0
            END
        )::DOUBLE
            / NULLIF(COUNT(*), 0)
            AS invalid_rate

    FROM core.fact_order_item

    GROUP BY item_status

    ORDER BY invalid_items DESC
    """
).df()

display(invalid_by_status)

,item_status,total_items,invalid_items,invalid_rate
0,Shipped,54456,16307.0,0.299453
1,Complete,45422,13577.0,0.298908
2,Returned,18041,5556.0,0.307965
3,Processing,35871,0.0,0.000000
4,Cancelled,27072,0.0,0.000000


In [83]:
#Check whether the anomaly exists in the raw source
source_comparison = connection.execute(
    """
    SELECT
        r.id AS raw_order_item_id,

        r.created_at AS raw_created_at,
        r.shipped_at AS raw_shipped_at,
        r.delivered_at AS raw_delivered_at,
        r.returned_at AS raw_returned_at,

        s.created_at AS staging_created_at,
        s.shipped_at AS staging_shipped_at,
        s.delivered_at AS staging_delivered_at,
        s.returned_at AS staging_returned_at,

        f.order_item_created_at AS fact_created_at,
        f.shipped_at AS fact_shipped_at,
        f.delivered_at AS fact_delivered_at,
        f.returned_at AS fact_returned_at

    FROM raw.order_items AS r

    INNER JOIN stg.order_items AS s
        ON r.id = s.order_item_id

    INNER JOIN core.fact_order_item AS f
        ON s.order_item_id = f.order_item_id

    WHERE f.valid_timeline_flag = 0

    LIMIT 20
    """
).df()

display(source_comparison)

,raw_order_item_id,raw_created_at,raw_shipped_at,raw_delivered_at,raw_returned_at,staging_created_at,staging_shipped_at,staging_delivered_at,staging_returned_at,fact_created_at,fact_shipped_at,fact_delivered_at,fact_returned_at
0,129627,2023-10-28 01:49:38.000000,2023-10-27 08:52:00.000000,2023-10-30 11:06:00.000000,NaT,2023-10-28 01:49:38.000000,2023-10-27 08:52:00.000000,2023-10-30 11:06:00.000000,NaT,2023-10-28 01:49:38.000000,2023-10-27 08:52:00.000000,2023-10-30 11:06:00.000000,NaT
1,132606,2024-04-28 16:57:50.000000,2024-04-27 07:29:00.000000,2024-05-02 01:13:00.000000,NaT,2024-04-28 16:57:50.000000,2024-04-27 07:29:00.000000,2024-05-02 01:13:00.000000,NaT,2024-04-28 16:57:50.000000,2024-04-27 07:29:00.000000,2024-05-02 01:13:00.000000,NaT
2,138256,2024-03-14 13:05:47.000000,2024-03-13 17:01:00.000000,2024-03-16 09:57:00.000000,NaT,2024-03-14 13:05:47.000000,2024-03-13 17:01:00.000000,2024-03-16 09:57:00.000000,NaT,2024-03-14 13:05:47.000000,2024-03-13 17:01:00.000000,2024-03-16 09:57:00.000000,NaT
3,138874,2023-11-08 10:33:37.000000,2023-11-06 15:47:00.000000,2023-11-07 16:14:00.000000,NaT,2023-11-08 10:33:37.000000,2023-11-06 15:47:00.000000,2023-11-07 16:14:00.000000,NaT,2023-11-08 10:33:37.000000,2023-11-06 15:47:00.000000,2023-11-07 16:14:00.000000,NaT
4,139190,2023-08-16 13:23:27.000000,2023-08-16 08:29:00.000000,2023-08-19 12:30:00.000000,NaT,2023-08-16 13:23:27.000000,2023-08-16 08:29:00.000000,2023-08-19 12:30:00.000000,NaT,2023-08-16 13:23:27.000000,2023-08-16 08:29:00.000000,2023-08-19 12:30:00.000000,NaT
5,140702,2022-01-20 00:14:29.000000,2022-01-17 18:28:00.000000,2022-01-18 20:29:00.000000,NaT,2022-01-20 00:14:29.000000,2022-01-17 18:28:00.000000,2022-01-18 20:29:00.000000,NaT,2022-01-20 00:14:29.000000,2022-01-17 18:28:00.000000,2022-01-18 20:29:00.000000,NaT
6,140924,2022-01-06 02:45:27.000000,2022-01-04 08:29:00.000000,2022-01-09 06:12:00.000000,NaT,2022-01-06 02:45:27.000000,2022-01-04 08:29:00.000000,2022-01-09 06:12:00.000000,NaT,2022-01-06 02:45:27.000000,2022-01-04 08:29:00.000000,2022-01-09 06:12:00.000000,NaT
7,141928,2022-05-21 22:56:38.000000,2022-05-19 20:15:00.000000,2022-05-24 17:32:00.000000,NaT,2022-05-21 22:56:38.000000,2022-05-19 20:15:00.000000,2022-05-24 17:32:00.000000,NaT,2022-05-21 22:56:38.000000,2022-05-19 20:15:00.000000,2022-05-24 17:32:00.000000,NaT
8,143208,2019-06-29 11:24:51.000000,2019-06-27 23:20:00.000000,2019-07-02 06:18:00.000000,NaT,2019-06-29 11:24:51.000000,2019-06-27 23:20:00.000000,2019-07-02 06:18:00.000000,NaT,2019-06-29 11:24:51.000000,2019-06-27 23:20:00.000000,2019-07-02 06:18:00.000000,NaT
9,146480,2024-02-17 16:24:15.000000,2024-02-17 10:01:00.000000,2024-02-22 06:51:00.000000,NaT,2024-02-17 16:24:15.000000,2024-02-17 10:01:00.000000,2024-02-22 06:51:00.000000,NaT,2024-02-17 16:24:15.000000,2024-02-17 10:01:00.000000,2024-02-22 06:51:00.000000,NaT


In [84]:
#Compare item creation with order creation
order_vs_item_timeline = connection.execute(
    """
    SELECT
        COUNT(*) AS items,

        SUM(
            CASE
                WHEN i.shipped_at < i.created_at
                THEN 1 ELSE 0
            END
        ) AS shipped_before_item_created,

        SUM(
            CASE
                WHEN i.shipped_at < o.created_at
                THEN 1 ELSE 0
            END
        ) AS shipped_before_order_created

    FROM stg.order_items AS i

    INNER JOIN stg.orders AS o
        ON i.order_id = o.order_id
    """
).df()

display(order_vs_item_timeline)

,items,shipped_before_item_created,shipped_before_order_created
0,180862,35440.0,0.0
